# Podcast — *After the Spike* (Dean Spears & Michael Geruso, 2025)

14 áudios MP3 em pt-BR, voz clonada via **OmniVoice**, cobrindo o livro inteiro: prólogo + 12 capítulos + apêndice. Substitui a leitura do livro.

Cada episódio tem ~1100-1600 palavras (≈ 7-10 min de áudio). Total ~2h30 — equivalente a destilar um livro de ~80 mil palavras.

Mesmo padrão de pipeline do `Podcast_Prepara_OmniVoice.ipynb` (case Contabilizei): mesma `voz.mp3`, fingerprint MD5 da voz no cache, rsync entre Colab e Drive.

**Pré-requisitos (uma vez):**
1. Runtime → Change runtime type → **T4 GPU**.
2. `voz.mp3` (~30s, mono, 16-24kHz) em `MyDrive/omnivoice/voz.mp3`.
3. Pasta `MyDrive/AfterTheSpike_Podcasts/` é criada automaticamente.

**O que esse notebook faz:**
- Instala OmniVoice + ffmpeg (sem TeXLive/Manim — pipeline leve)
- Carrega os 14 scripts já embutidos abaixo (nada pra subir)
- Para cada podcast: chunking por parágrafo, 1 wav por chunk, concatena com pausa de 0.4s
- Exporta MP3 (libmp3lame 96 kbps) por podcast
- Empacota num `after_the_spike_podcasts.zip` + download direto

**Tempo estimado:** ~25-40 min em T4 (14 episódios). Cache por hash do trecho + fingerprint MD5 da voz, então re-executar é barato.


## 1. Verificar GPU

In [1]:
!nvidia-smi

Fri May 15 15:09:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Instalar dependências

Primeira execução: ~3-5 min (OmniVoice + pesos baixam na célula 6).

In [2]:
# 0) Limpa estado do apt (Colab às vezes tem dpkg interrompido + repo r2u quebrado)
!sudo dpkg --configure -a 2>&1 | tail -3
!sudo rm -f /etc/apt/sources.list.d/r2u.list /etc/apt/sources.list.d/cran*.list 2>/dev/null

# 1) Só ffmpeg + libsndfile
!apt-get -qq update 2>&1 | tail -3
!apt-get -qq install -y --fix-missing ffmpeg libsndfile1 2>&1 | tail -3

# 2) Toolchain Python atualizado
%pip install -q --upgrade pip setuptools wheel

# 3) OmniVoice (voice cloning)
import importlib.util
if importlib.util.find_spec('omnivoice') is None:
    print('→ instalando OmniVoice (~2-4 min)...')
    %pip install -q git+https://github.com/k2-fsa/OmniVoice.git
else:
    print('✓ OmniVoice já presente')

# 4) Reconcilia numpy (faixa OBRIGATÓRIA >=2.1,<2.2 — mesmo do notebook de produção)
%pip install -q --upgrade --force-reinstall --no-deps 'numpy>=2.1,<2.2'
%pip install -q soundfile pydub

# 5) Sanity check
import subprocess, sys
chk = subprocess.run(
    [sys.executable, '-c', 'import numpy; import soundfile; import torch; import torchaudio'],
    capture_output=True, text=True,
)
if chk.returncode != 0:
    print('❌ stack inconsistente:')
    print(chk.stderr[-800:])
    print('\n⚠ Faça Runtime → Restart session e rode esta célula de novo.')
else:
    print('\n✓ numpy / soundfile / torch / torchaudio OK')

import numpy
print(f'   numpy carregado: {numpy.__version__}  (esperado: 2.1.x)')
print(f'   se for 2.0.x ou 2.2.x → Runtime → Restart session e rode esta célula DE NOVO.')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✓ OmniVoice já presente

✓ numpy / soundfile / torch / torchaudio OK
   numpy carregado: 2.1.3  (esperado: 2.1.x)
   se for 2.0.x ou 2.2.x → Runtime → Restart session e rode esta célula DE NOVO.


## 3. Montar Google Drive (pra pegar a voz de referência)

In [3]:
!fusermount -u /content/drive 2>/dev/null || true
!umount -l /content/drive 2>/dev/null || true
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 4. Configurar caminhos

- `REF_AUDIO`: voz de referência (`voz.mp3`) — mesmo path dos outros notebooks OmniVoice.
- `REF_AUDIO_TAG`: fingerprint MD5 da voz, entra no nome de cada WAV cacheado.
- `OUTPUT_DIR`: pasta no Drive onde MP3s, cache de WAVs e zip final ficam.
- `LOCAL_OUT`: cópia local rápida.


In [4]:
import os, hashlib

REF_AUDIO       = "/content/drive/MyDrive/omnivoice/voz.mp3"
OUTPUT_DIR      = "/content/drive/MyDrive/AfterTheSpike_Podcasts"
DRIVE_WAV_CACHE = os.path.join(OUTPUT_DIR, "wavs_cache")
LOCAL_OUT       = "/content/podcasts_out"
LOCAL_WAV_CACHE = "/content/cache_wavs"
GAP_SEC         = 0.4
MP3_BITRATE     = "96k"

if not os.path.exists(REF_AUDIO):
    raise FileNotFoundError(
        f"❌ voz de referência não encontrada: {REF_AUDIO}\n"
        f"   Suba voz.mp3 (~30s, mono, 16-24kHz) pra esse caminho no Drive."
    )

with open(REF_AUDIO, "rb") as _f:
    REF_AUDIO_TAG = hashlib.md5(_f.read()).hexdigest()[:8]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_WAV_CACHE, exist_ok=True)
os.makedirs(LOCAL_OUT, exist_ok=True)
os.makedirs(LOCAL_WAV_CACHE, exist_ok=True)

print(f"✓ REF_AUDIO:       {REF_AUDIO}")
print(f"✓ REF_AUDIO_TAG:   {REF_AUDIO_TAG}")
print(f"✓ OUTPUT_DIR:      {OUTPUT_DIR}")
print(f"✓ DRIVE_WAV_CACHE: {DRIVE_WAV_CACHE}")
print(f"✓ LOCAL_OUT:       {LOCAL_OUT}")


✓ REF_AUDIO:       /content/drive/MyDrive/omnivoice/voz.mp3
✓ REF_AUDIO_TAG:   4a4b0491
✓ OUTPUT_DIR:      /content/drive/MyDrive/AfterTheSpike_Podcasts
✓ DRIVE_WAV_CACHE: /content/drive/MyDrive/AfterTheSpike_Podcasts/wavs_cache
✓ LOCAL_OUT:       /content/podcasts_out


## 5. Scripts dos 14 episódios (embutidos — nada pra subir)

In [5]:
PODCASTS = [
    {
        "id": "01_prologo",
        "title": "Prólogo — o livro que pede pra você pensar de novo",
        "text": """Episódio um. Prólogo: o livro que pede pra você pensar de novo.

Antes de tudo, uma frase que sustenta o livro inteiro: a humanidade está num caminho de despovoamento. Não de população menor e estável. De despovoamento.

Pare por um segundo. Você, seus pais, seus avós, qualquer ancestral cujo nome você conhece, todos viveram dentro de uma população que cresceu. Sempre cresceu. Agora, pela primeira vez em séculos, isso vai se inverter. As taxas de natalidade estão caindo no mundo todo, e a inversão está no horizonte. O detalhe, e esse é o ponto que assusta os autores, é que quando a população começar a encolher, ela não para sozinha em seis bilhões, nem em quatro, nem em dois. Cada geração nasce menor que a anterior, e o efeito é exponencial. É isso que Dean Spears e Michael Geruso, os autores de After the Spike, chamam de despovoamento.

Esse é o ponto de partida desta série. Vamos percorrer o livro inteiro, capítulo por capítulo, pra você não precisar lê-lo. Se eu fizer meu trabalho direito, no fim você vai ter os argumentos, os números e os insights mais fortes, e vai conseguir formar sua própria opinião. Hoje, o prólogo.

Quem são os autores. Dean Spears e Michael Geruso são economistas, professores da Universidade do Texas. Os dois se conheceram em Princeton, dividindo uma sala de doutorado. A área deles não é juros, nem bolsa. É gente. É bem-estar humano. Spears mora boa parte do tempo em Uttar Pradesh, no norte da Índia, onde toca uma ONG chamada r-i-c-e, que trabalha com bebês prematuros e abaixo do peso em hospitais públicos. Geruso passou um tempo na Casa Branca, assessorando o Conselho de Conselheiros Econômicos sobre tendências populacionais. Ou seja, não são opinionistas de gabinete. Eles passaram a carreira inteira olhando pra dados de nascimento, morte, saúde infantil e poluição. E a tese desse livro nasceu de uma pergunta simples que eles passaram a se fazer: o que acontece quando você zoom out e olha pra esses números na escala da humanidade inteira, em séculos?

A resposta deles é desconfortável pros dois lados do debate. Primeiro, eles dizem que o discurso de que somos pessoas demais e que reduzir a população resolveria mudança climática e desigualdade é falso. Mas, segundo, e aqui o livro fica realmente interessante, eles dizem que a fantasia oposta também é furada. A ideia de que, beleza, vamos cair pra uns bilhões e estabilizar num mundo mais leve, mais sustentável, isso não acontece automaticamente. Estabilizar exige escolha, esforço, política pública. Não é o cenário-base. O cenário-base é cair pra sempre.

Eles organizam o livro em torno de três afirmações grandes. Vale anotar essas três, porque é o esqueleto da nossa série inteira.

Afirmação um, da Parte um do livro: nenhum futuro é mais provável do que o futuro em que as pessoas no mundo todo escolham ter filhos abaixo do nível de reposição, e que isso se mantenha. E reposição, pra você guardar, é em torno de dois filhos e um décimo por mulher. Abaixo disso, cada geração é menor que a anterior. E praticamente o planeta inteiro já está abaixo disso. Coréia do Sul, Itália, Espanha, China, Brasil, Índia. Mesmo lugares onde a fertilidade ainda é alta, como partes da África subsaariana, estão caindo rápido.

Afirmação dois, das Partes dois e três do livro: uma população mundial estabilizada seria, no geral, melhor do que uma população em despovoamento contínuo. Essa frase parece pequena, mas é a tese moral do livro. Eles vão ter que defender que mais gente, vivendo bem, é melhor do que menos gente vivendo bem. E vão ter que enfrentar o argumento ambiental, o argumento de desigualdade, o argumento de bem-estar. A gente vai destrinchar isso nos próximos episódios.

Afirmação três, da Parte quatro: ninguém ainda sabe como estabilizar um mundo que está despovoando. Olha que humildade. Eles não te entregam a receita. Mas eles dizem o seguinte: a humanidade já fez transições gigantes antes, do feudalismo pro capitalismo industrial, da medicina pré-antibiótica pra expectativa de vida atual. Se a gente decidir que estabilizar importa, dá pra tentar. Mas precisa decidir.

Tem um ponto sensível que merece ser dito agora, no episódio um, pra não ficar pairando. Spears e Geruso são explícitos: nada do que eles defendem envolve retroceder em direitos reprodutivos, direito ao aborto, ou equidade de gênero. Eles repetem isso várias vezes ao longo do livro. A pergunta não é se a mulher tem o direito de escolher quantos filhos ter. Tem. A pergunta é se a sociedade, do jeito que está organizada hoje, dá condições reais pra quem quer ter filhos conseguir ter. Spoiler: não dá. Cuidar de criança custa carreira, custa sono, custa rede de apoio, custa dinheiro. E é isso que eles querem discutir.

Tem uma pergunta filosófica também, e essa é a mais difícil. Como você valoriza uma vida que ainda não existe? Se uma criança nunca chega a ser concebida, ela não está sofrendo, ela simplesmente não é nada. Então faz diferença se mais vidas boas, vidas como a sua ou melhores, são vividas no futuro? Ou tanto faz? Esse é um debate antigo da filosofia, chamado ética populacional, e o livro vai ter um apêndice inteiro só sobre isso, sobre a chamada conclusão repugnante do filósofo Derek Parfit. A gente vai chegar lá no fim da série.

Pra fechar o prólogo, três coisas que eu acho que vale você levar pro próximo episódio. Primeira: o livro não diz que é uma crise pra resolver amanhã. Diz que tem algumas décadas. O pico populacional global deve acontecer por volta de dez bilhões, em algum momento entre dois mil e oitenta e dois mil e noventa, e depois começa a curva descendente. Tem tempo de pensar, e justamente por isso eles querem que essa conversa não seja sequestrada pelos extremos. Quem está falando mais alto hoje sobre baixa natalidade no Ocidente costuma ser quem quer empurrar agenda nacionalista, antiimigração, ou de controle do corpo das mulheres. Spears e Geruso querem ocupar esse espaço com dados e com uma ética mais generosa.

Segunda coisa: a tensão central do livro é essa. Ter filho é escolha pessoal, ponto. Mas a soma das escolhas pessoais tem consequência coletiva. Igual escolha de consumo, escolha de poluir, escolha de votar. Não dá pra dizer que cada um faz o que quer e a conta não chega na sociedade. Chega. A grande pergunta do livro é como cuidar melhor uns dos outros, sem violar liberdade.

E terceira: o título do livro, After the Spike, depois do pico. A imagem que eles querem que você carregue é a de uma agulha gigantesca no gráfico da humanidade. Nascemos numa subida vertical de população, de menos de um bilhão pra oito bilhões em duzentos anos. Esse é o pico. Daqui pra frente, ou a gente fica do outro lado dele caindo pra sempre, ou a gente segura num platô. É essa escolha que o livro propõe.

No próximo episódio, capítulo um, o pico em si. Como chegamos aqui, por que a curva é tão íngreme, e por que ela está prestes a virar. Até lá.""",
    },
    {
        "id": "02_o_pico",
        "title": "Cap. 1 — O pico",
        "text": """Episódio dois. O pico.

No episódio anterior, contei o argumento geral do livro: a humanidade está num caminho de despovoamento, não de estabilização. Hoje a gente entra no primeiro capítulo de verdade. Ele se chama The Spike, o pico. E começa com um dado que dá vertigem.

Dois mil e doze. Cento e quarenta e seis milhões de bebês nasceram naquele ano. Foi mais do que em qualquer ano anterior na história humana. E também foi mais do que em qualquer ano depois. Spears e Geruso defendem que dois mil e doze pode ter sido o pico de nascimentos da humanidade. Não desta década. Não deste século. De toda a história. Pra sempre. Nenhuma projeção demográfica séria espera algo diferente.

Pra entender essa frase, você precisa visualizar a curva da população humana ao longo do tempo. Há dez mil anos, éramos uns cinco milhões. A região metropolitana de Atlanta, hoje, é maior. Há mil anos, um quarto de bilhão. Em mil e oitocentos, há duzentos anos, passamos do primeiro bilhão. Uma a cada cinco pessoas que já viveram na história nasceu depois de mil e oitocentos. Isso é a subida do pico. Foi rapidíssima em escala histórica. E está prestes a acabar.

Quando você desenha esse gráfico, ele parece uma agulha cravada no tempo. Sobe quase vertical nos últimos dois séculos, chega num topo entre dois mil e sessenta e dois mil e noventa, dependendo da projeção, e começa a descer. Spears e Geruso chamam essa curva de The Spike. O Pico. Não é coincidência o livro se chamar After the Spike. A pergunta deles é: o que acontece depois.

Quem faz as projeções? Três grupos: a ONU, o Instituto Internacional de Análise de Sistemas Aplicados na Áustria, e o Instituto de Métricas e Avaliação em Saúde da Universidade de Washington. As datas variam, oitenta, oitenta e poucos, sessenta e poucos. Mas, em escala humana, vinte anos pra cá ou pra lá é a mesma coisa. Todos os três concordam: o pico acontece nesse século. Todos os três projetam que a fertilidade segue caindo.

Agora a parte mais surpreendente. Se a fertilidade ficar onde está hoje, ou seja, sem cair mais, só travada onde já está, ainda assim a humanidade despovoa. Por quê? Porque a média global já é, ou está prestes a ser, abaixo de dois filhos por casal. E abaixo de dois é a linha crítica. Cada geração nasce menor que a anterior. É decaimento exponencial.

Olha a lista de países que já estão abaixo de dois: Estados Unidos com um vírgula seis. México. Canadá. Brasil. Rússia. Tailândia. União Europeia inteira em um vírgula cinco. China. Índia. Os dois países mais populosos do mundo, China e Índia, abaixo de dois. Em todos os estados americanos. Em todas as províncias canadenses. Entre brancos, negros e hispânicos nos Estados Unidos, separadamente. Não tem mais um grupo grande de exceção. A norma virou outra.

E a norma é nova. Em duas terças partes do planeta, as pessoas já vivem em países com fertilidade abaixo do nível de reposição.

Aí o livro faz uma coisa que eu achei muito boa pedagogicamente. Ele te apresenta a Preeti. Preeti é uma mulher real, indiana, mora num vilarejo do estado de Uttar Pradesh, norte da Índia. Em dois mil e vinte e dois ela teve a primeira filha num hospital público lotado. A bebê nasceu muito pequena, abaixo do peso. Foi pra um programa chamado Mãe Canguru, que mantém o bebê em contato pele a pele com a mãe pra regular temperatura. Programa esse que a ONG do Spears ajuda a financiar.

Preeti vive numa casa metade barro, metade tijolo. Não terminou o ensino médio. É pobre, casou nova. E Preeti diz, com naturalidade, que quer ter mais um filho, e que se for menino, ela vai fazer a cirurgia. Sterilização. Dois filhos é o suficiente.

Por que isso importa? Porque Preeti é o ponto médio da humanidade. Renda média, expectativa de vida média, fertilidade média. Se uma mulher pobre, sem muita escolaridade, no estado mais populoso da Índia, planeja ter dois filhos, isso significa que, no mundo todo, a média é menor que dois. E entre as mulheres indianas com ensino médio, que é uma fração crescente, a média já é um vírgula oito. Igual à média americana.

A revolução demográfica não está na bolha rica de Tóquio ou Milão. Está acontecendo no vilarejo da Preeti.

Tem um detalhe matemático que eu preciso te mostrar, porque é dele que sai o título do livro. Spears e Geruso fazem uma conta no capítulo: se a humanidade seguir nessa trajetória, vai chegar a uns cento e cinquenta bilhões de nascimentos no total. Hoje, somando todos os humanos que já existiram desde a aurora da espécie, são cento e vinte bilhões. Ou seja, se o caminho atual continuar, a história humana já está quatro quintos escrita. Quatro de cinco humanos que vão existir um dia já existiram.

Pause aí. Quatro quintos. Isso é uma forma poética de dizer que a esmagadora maioria da experiência humana, em número absoluto, já aconteceu. Se continuarmos assim.

Outra pergunta importante que o capítulo enfrenta: a queda não pode dar uma virada sozinha? Tipo, as pessoas voltarem a ter mais filhos por algum motivo natural? Resposta dos autores: não tem precedente. Desde mil novecentos e cinquenta, vinte e seis países entraram em fertilidade abaixo de um vírgula nove. Vinte e seis. Nenhum deles, nem um, voltou pra um nível de reposição. Zero a vinte e seis. Não no Canadá, não no Japão, não na Escócia, não em Taiwan. Em todos esses, os governos acham que têm políticas de incentivo. E mesmo assim, ficou abaixo de dois. Pra sempre, até agora.

Ou seja: estabilizar a partir do despovoamento exigiria uma virada que nunca aconteceu na história moderna registrada. Pode acontecer. Mas o ônus da prova está com quem aposta nisso.

A última imagem do capítulo é uma analogia muito boa. Spears e Geruso lembram o leitor: quando você ouviu pela primeira vez que mudança climática era pra valer? Provavelmente nos anos noventa. Mas o Congresso americano ouviu cientistas sobre isso já nos anos cinquenta. Lyndon Johnson, em sessenta e cinco, falou sobre dióxido de carbono em discurso oficial. A Casa Branca emitiu um relatório chamando CO dois de poluente em sessenta e cinco. Foi cedo? Foi. Mas se ninguém tivesse começado em sessenta e cinco, a política climática de hoje não teria as ferramentas que tem.

A analogia é direta. O pico do despovoamento talvez esteja a sessenta anos da gente. E os autores estão dizendo: é agora que começa essa conversa. Não pra resolver amanhã, mas pra que daqui sessenta anos a humanidade tenha mais do que um problema gigante e nenhuma ideia do que fazer.

No próximo episódio, capítulo dois. A linha divisória entre crescimento e decadência. Como exatamente dois filhos por casal vira o destino do planeta. Tem matemática, mas eu prometo que vai ficar claro. Te espero lá.""",
    },
    {
        "id": "03_linha_divisoria",
        "title": "Cap. 2 — A linha divisória entre crescimento e decadência",
        "text": """Episódio três. A linha divisória entre crescimento e decadência.

No episódio anterior, conheci você ao pico, the spike, e à Preeti, a mãe indiana que quer dois filhos. Hoje, capítulo dois do livro. E pra abrir esse capítulo eu quero te apresentar duas irmãs.

Seema e Reema. Enfermeiras no mesmo hospital de Uttar Pradesh onde a Preeti deu à luz. As duas vêm de um vilarejo rural. Pai e mãe tiveram oito filhos. O pai delas, ele mesmo, era um de oito irmãos. Há uma geração, oito filhos não era estranho. Os pais sacrificaram décadas pra educar os oito. As duas conseguiram diploma. Furaram a estatística.

Reema, a irmã mais velha, foi a primeira a virar enfermeira profissional. Hoje ela diz: dois filhos. Com certeza dois.

Seema, vinte e dois anos, ainda solteira, casamento arranjado por vir, diz outra coisa. Um filho. Só um. Ela ama a família grande em que cresceu, mas pra si mesma, ela não quer isso. O tempo é outro, ela diz.

Spears e Geruso pegam essa diferença, dois pra Reema, um pra Seema, e fazem uma observação aparentemente boba. Ouve só: um vírgula nove é menor que dois vírgula zero. Eles chamam isso de uma verdade profunda. E parece bobagem. Mas é o ponto inteiro do capítulo. Porque dois é a linha. Acima de dois, a humanidade cresce exponencialmente. Abaixo de dois, ela decresce exponencialmente. Não existe estabilidade abaixo de dois. Não existe planalto. É decadência composta.

Aqui entra a primeira lição matemática do capítulo, e prometo manter leve. Crescimento populacional e decadência populacional seguem a mesma matemática. Exponencial pros dois lados. Nos últimos cem anos, a humanidade quadruplicou. Nos últimos duzentos anos, multiplicou por oito. E a álgebra que fez isso pra cima é a mesma que pode fazer pra baixo. Em duzentos anos, podemos cair pelo mesmo fator.

A segunda lição é por que dois é a linha. Em geral, metade das crianças nasce mulher. Pra cada duas mulheres adultas hoje, se cada uma tem dois filhos, a próxima geração terá duas mulheres adultas. Mesmo número, geração após geração. Estabilidade.

Mas se cada mulher tem em média um vírgula cinco filho, e isso é a média europeia atual, ela tem em média zero vírgula sete e cinco filhas. Cem mulheres dessa geração viram setenta e cinco mulheres na próxima. Setenta e cinco viram cinquenta e seis. Cinquenta e seis viram quarenta e dois. Pra cada sete gerações a partir de agora, com essa fertilidade, cento e trinta e dois milhões de bebês por ano viram nove milhões. Em catorze gerações, um milhão. Em vinte e uma gerações, cento e sessenta mil meninas no ano todo no planeta inteiro. Isso é menos do que nasce no Texas hoje.

Tem um número técnico que eu preciso te mencionar: muita gente fala em dois vírgula um como nível de reposição. Por que esse decimal? Porque nem todo bebê chega à idade adulta. Tem mortalidade infantil. Em locais onde metade dos bebês morria antes dos cinco, o nível de reposição era próximo de quatro. No Reino Unido no ano mil e novecentos, era dois vírgula seis. Em Uttar Pradesh nos anos dois mil e dez, era dois vírgula trinta e nove. Em Kerala, estado mais rico da Índia, dois vírgula zero nove.

Mas, como o livro diz, o futuro vai ter mortalidade infantil próxima de zero em quase todo lugar. Então o nível de reposição global, no futuro, converge pra perto de dois. Os autores adotam dois como aproximação suficientemente boa. Eu também vou adotar nessa série.

Agora vem o ponto que provavelmente mais te surpreende. A taxa de natalidade está caindo há séculos. Há séculos. Não começou com pílula, não começou com feminismo, não começou com industrialização. Vem caindo há tanto tempo quanto existem dados pra medir. Em mil e oitocentos, mulheres brancas casadas nos Estados Unidos tinham em média sete filhos. Na média global, seis. Em mil e novecentos, a média americana caiu pra quatro. Na França, abaixo de três. Globalmente, cinco vírgula cinco. Em mil novecentos e cinquenta, ainda cinco em média no mundo. Hoje, dois vírgula três.

Espera. Se a taxa de natalidade está caindo há tanto tempo, como a população explodiu nos últimos duzentos anos?

Resposta: porque a mortalidade infantil despencou ainda mais rápido. A população não cresceu porque as pessoas começaram a ter mais filhos. Cresceu porque os filhos pararam de morrer.

Esse é um dos fatos centrais do livro, e tem uma elegância dolorosa. Pense na família do Benjamin Franklin. Ele tinha dezessete irmãos. Quatro morreram na infância. Mortalidade infantil de uns vinte e quatro por cento. Naquele tempo, era considerada sorte. Em alguns lugares dos mil setecentos, entre um terço e metade dos bebês morria antes dos cinco anos.

Hoje, na Austrália, Alemanha ou Coréia do Sul, de duzentas e cinquenta crianças que nascem, duzentas e quarenta e nove chegam aos cinco. Globalmente, a chance de morrer na infância é menos de quatro por cento.

Esse foi o que o economista Angus Deaton, ganhador do Nobel e ex-orientador dos autores, chamou de A Grande Fuga. Foi a humanidade fugindo da morte precoce. Aquela imagem clássica de que antigamente as pessoas morriam aos trinta? Não é bem assim. Quem chegava na vida adulta vivia até os sessenta, setenta. O que melhorou drasticamente foi a sobrevivência infantil. Quem fica velho hoje fica mais ou menos tão velho quanto o velho de mil setecentos. O que mudou é que muita gente que morreria criança hoje chega lá.

Aí vem o ponto que muda tudo pro futuro. A grande fuga acabou. Não tem mais pra onde a mortalidade infantil cair. Já está perto do piso. Em Uttar Pradesh, o lugar pobre da Índia onde Spears trabalha, a mortalidade infantil hoje é menor do que era nos Estados Unidos em mil e oitocentos. O motor que empurrou a humanidade pra cima do pico, mortalidade infantil caindo, esgotou.

Então, daqui pra frente, o que define a trajetória populacional é uma coisa só: a fertilidade. E a fertilidade está bem abaixo de dois.

Os autores antecipam três objeções comuns. Vou listar.

Primeira: e Finlândia entre dois mil e dois mil e dez? A fertilidade subiu nessa década específica. Não é prova de que pode subir? Resposta dos autores: dá zoom out. Em dois mil e vinte, a Finlândia já tinha caído pra um vírgula quatro. Em mil novecentos e sessenta era dois vírgula sete. Tem ruído de curto prazo, claro, mas o sinal de longo prazo é claríssimo.

Segunda: e os lugares onde a fertilidade ainda é bem acima de dois, como a África subsaariana? Resposta: a África subsaariana, há quarenta anos, tinha fertilidade de seis vírgula oito. Hoje está em quatro vírgula dois. Cai um por cento ao ano. Todas as equipes sérias de projeção esperam que continue caindo.

Terceira: e se a expectativa de vida dobrar ou quadruplicar? Não compensaria? Resposta surpreendente: não. Se a média ainda for menor que dois filhos por mulher, ainda que cada um viva trezentos anos, a quantidade de bebês ainda cai exponencialmente. As pessoas vão acumular anos de vida, mas o tamanho da próxima geração ainda encolhe. Pra estabilizar, tem que voltar a ter dois filhos. Não tem outro caminho.

Tem uma analogia bonita no fim do capítulo. Quando alguém joga uma bola pra cima, ela sobe, atinge um ápice, e volta. Não fica suspensa no topo. A mesma força que desacelerou a subida puxa pra baixo. Muita gente, inclusive economistas famosos, supôs que a fertilidade ia chegar em dois e parar. Não tem por que. Dois é só um número. Não tem ímã.

Estatísticos da Universidade de Washington estimaram em dois mil e vinte e três: há noventa por cento de chance de que a fertilidade global fique abaixo de dois pelo menos até o ano de dois mil e trezentos. Noventa por cento.

Resumo do capítulo dois: a humanidade chegou ao pico porque mortalidade caiu, não porque natalidade subiu. A mortalidade já caiu quase tudo o que podia. Agora só sobra a natalidade pra mexer o ponteiro. E a natalidade está claramente abaixo da linha de reposição. Pra estabilizar, alguém precisa decidir e trabalhar pra isso acontecer.

Aí termina a Parte um do livro. Próximo episódio, capítulo três, começa a Parte dois, que é o argumento contra mais gente. Aquele clássico: mas e o planeta? Pessoas a mais não destroem o meio ambiente? Vou te contar como os autores respondem essa pergunta. Te espero lá.""",
    },
    {
        "id": "04_pessoas_e_planeta",
        "title": "Cap. 3 — O que as pessoas fazem ao planeta",
        "text": """Episódio quatro. O que as pessoas fazem ao planeta.

Os dois primeiros capítulos do livro foram sobre como chegamos aqui e por que o despovoamento é provável. A partir de agora, começa a Parte dois, que os autores chamam de o argumento contra mais pessoas. E o primeiro grande argumento é o ambiental. Mais gente é mais poluição, mais aquecimento, menos planeta. Reduzir a população é, no senso comum, ambientalmente bom. Spears e Geruso vão contestar isso, e eles começam o capítulo três com uma história que parece um paradoxo.

Dois mil e treze. China. Crise de poluição em Pequim chamada de apocalipse do ar. A embaixada americana mediu setecentos e cinquenta e cinco numa escala que normalmente vai até quinhentos. O ar mata. Nos dez anos seguintes, a China acrescentou cinquenta milhões de pessoas à sua população. Cinquenta milhões. Mais ou menos uma Espanha inteira, ou uma Argentina, ou uma Uganda. Pela lógica do senso comum, a poluição devia ter explodido.

Aconteceu o contrário. A poluição por material particulado na China caiu pela metade nesse período. Mais gente, menos poluição. Como?

A resposta é simples e essencial pro capítulo inteiro. O que importa não é quantas pessoas existem. É o que essas pessoas fazem. É o que os governos delas regulam. É qual energia é usada. A relação entre tamanho da população e dano ambiental nunca foi linear.

Pra fixar isso na cabeça do leitor, Spears e Geruso fazem um exercício imaginativo lindo. Imagine duas opções de moradia. A primeira: uma cabana rústica na floresta, lareira a lenha, água de nascente, vista pra montanha, cheiro de pinheiro, coruja na varanda. A segunda: apartamento no quarto andar de um prédio comum, com parede compartilhada de dois lados, teto e chão dividindo com vizinhos, ar-condicionado, fumaça do beco, pombo no toldo, ratos na escada do metrô.

Qual dos dois é mais ecológico?

Intuição diz cabana. Mais natureza. Resposta correta: apartamento. De longe. A cabana perde calor pra todos os lados, queima madeira que vira material particulado, depende de eletricidade que percorre quilômetros de linha de transmissão com perdas, e exige carro pra tudo. O apartamento aquece quatro paredes compartilhadas, gasta menos energia por morador, e a vida densa diminui o consumo per capita pra praticamente tudo. A intuição de proximidade com a natureza confunde com proximidade com baixo impacto.

Esse é o avesso que o livro quer instalar na sua cabeça antes de entrar nos dados duros.

Agora, vamos aos dados. Spears e Geruso fazem uma lista das grandes ameaças ambientais que a humanidade enfrentou no último meio século. Chumbo no ar nos anos setenta. Camada de ozônio nos anos oitenta. Chuva ácida nos anos noventa. Em todos esses casos, a população cresceu, e os problemas foram resolvidos. Não por menos pessoas, mas por regulação. Phase out do chumbo na gasolina. Protocolo de Montreal proibindo CFCs em mil novecentos e oitenta e sete. Limite de emissões de dióxido de enxofre nos Estados Unidos. Mais gente, e o ozônio começou a se recuperar mesmo assim. O dióxido de enxofre americano caiu noventa por cento desde mil novecentos e oitenta.

Mais um dado quase contraintuitivo. A exposição global ao material particulado, o tal PM dois vírgula cinco, está caindo desde dois mil e quinze. E nesses anos a humanidade acrescentou setecentos e cinquenta milhões de pessoas. Mais pessoas, menos poluição.

Quando os autores plotam densidade populacional contra poluição do ar, país a país, sai um nada. Sem correlação. Japão e Coreia do Sul são tão densos quanto a Índia, mas têm ar muito mais limpo. Níger é menos denso que os Estados Unidos, mas tem ar terrível. Por quê? Porque Níger queima resíduo agrícola e biomassa pra cozinhar, e o Japão tem energia limpa. O que mata o pulmão é coisa específica, não cabeça.

A cidade do México, que era uma das mais poluídas do mundo nos anos noventa, é hoje quase tão limpa quanto Roubaix, no norte da França. Não foi a população mexicana que caiu. Foi a política pública que mudou.

Agora, aqui, o livro encara o oponente histórico mais famoso: Paul Ehrlich e A Bomba Populacional, livro de mil novecentos e sessenta e oito. Ehrlich, professor de biologia de Stanford, vendeu mais de dois milhões de cópias prevendo fome em massa, escolha forçada entre controle populacional ou destruição da espécie. Tirou ele do Tonight Show de Johnny Carson como se fosse profeta. Sessenta anos depois, o futuro que ele dizia ser impossível, oito bilhões de pessoas, é o presente. E mesmo assim a mensagem dele, fewer people é fewer carbon, ainda permeia o senso comum.

Spears e Geruso usam o resto do capítulo pra detonar essa ideia com aritmética. Vou te contar o cálculo central, porque é importante.

Eles montaram um time multidisciplinar pra rodar modelos climáticos comparando dois futuros: o despovoamento clássico e o que eles chamam de estabilização. Pra ser justos com o argumento ambientalista, escolheram o cenário mais extremo de estabilização: doze bilhões de pessoas mantidas, plateaus pessimista pra descarbonização, ou seja, supondo que o mundo não vai chegar a emissão líquida zero nos próximos cem anos. Tudo pesado contra a tese deles.

Mesmo assim. Resultado: no cenário pessimista, o aquecimento global de longo prazo é de quatro vírgula vinte e dois graus se a humanidade despovoar, e quatro vírgula vinte e oito graus se estabilizar. Diferença de zero vírgula zero seis grau. Praticamente nada. Quatro bilhões de vidas extras pra esse aquecimento extra ridículo.

E no cenário onde o mundo cumpre o Acordo de Paris e estabiliza em dois graus? Dois vírgula zero seis graus pros dois caminhos. Sem diferença detectável até a terceira casa decimal.

Por que tão pouco? Porque o despovoamento é lento. Mesmo nas projeções mais agressivas, em dois mil e cinquenta a diferença entre os dois caminhos é menos de dois por cento da população global. As pessoas a mais só aparecem depois de dois mil e cem, e até lá a economia já estará majoritariamente descarbonizada. Zero vezes oito bilhões é igual a zero vezes nove bilhões. Quando a emissão por pessoa virar zero, o número de pessoas deixa de importar.

Tem uma frase do capítulo que vale repetir. Eles dizem o seguinte: a diferença entre dois graus e quatro graus é gigantesca. Mas essa diferença depende de política e tecnologia de descarbonização, não de população. A diferença que a população pode fazer é minúscula.

E olha o ponto que destrói a narrativa de Net Zero via menos filhos. Uma reportagem da NPR sugeriu que manter meninas na escola é uma estratégia contra mudança climática, porque escola atrasa filho e diminui fertilidade. Spears e Geruso concordam que manter meninas na escola é bom por mil motivos, mas o motivo climático não funciona. Por quê? Tempo. Uma criança que nasce hoje vai ter quinze anos em dois mil e quarenta. Quarenta anos em dois mil e sessenta e cinco. As metas do IPCC são pra dois mil e cinquenta. A demografia não trabalha nessa escala. Política e tecnologia trabalham.

Tem um dado bonito pra fechar. A pegada de carbono de longa duração de uma criança nascida hoje, no mundo todo, é menor do que a de uma criança nascida em mil novecentos e setenta. Por quê? Porque a tecnologia avançou e a sociedade vai descarbonizar ao longo da vida dessa criança. A jornalista Hannah Ritchie mostra que o britânico médio hoje emite menos da metade de CO dois do que emitia o britânico médio nos anos cinquenta. Apesar de ele ser, em alguns aspectos, mais consumista, esbanjador, ligado em aparelhos eletrônicos. A descarbonização venceu o consumo. A pegada de uma criança nascida hoje é menor que a sua. A de uma criança nascida em dez anos será menor ainda.

A conclusão do capítulo é firme. A humanidade tem desafios ambientais sérios e urgentes. Mas controle populacional não é uma ferramenta entre as urgentes pra resolver. Energia limpa é. Reflorestamento é. Regulação industrial é. Reduzir fertilidade não é. E pior, fingir que reduzir fertilidade vai resolver desvia atenção de quem precisa atacar o problema real.

Spears e Geruso reservaram um trecho pra falar do Bill McKibben, ambientalista enorme, autor de Maybe One, que defendeu em mil novecentos e noventa e oito que famílias menores podem ajudar o planeta. Eles dizem: McKibben mesmo escreveu que essa ideia tinha validade de cinquenta anos. Pra você, leitor, ler hoje como curiosidade histórica. Esses cinquenta anos já passaram. A janela em que população poderia ter sido alavanca ambiental fechou.

Próximo episódio, capítulo quatro. Um título provocativo: a população começa no corpo dos outros. É o capítulo sobre ética reprodutiva, coerção, abortos forçados, esterilizações forçadas, e a história suja das tentativas de reduzir natalidade. Te espero lá.""",
    },
    {
        "id": "05_corpos_dos_outros",
        "title": "Cap. 4 — A população começa no corpo dos outros",
        "text": """Episódio cinco. A população começa no corpo dos outros.

No episódio anterior, vimos que o argumento ambiental contra mais gente não se sustenta nos dados. Hoje, capítulo quatro, e o argumento muda completamente. Sai a biosfera, entra o corpo das mulheres.

A frase que dá título ao capítulo veio de uma colega cientista dos autores numa conferência. Spears e Geruso tinham apresentado a tese do pico. Ela levantou a mão e fez uma pergunta direta: mas quem é que vai ter os bebês, se as taxas de natalidade subirem? A resposta é mulheres. O ponto dela era que reprodução é desigual, e qualquer projeto de estabilização populacional toca, antes de tudo, na vida das mulheres. Esse capítulo é o livro encarando isso de frente.

A pergunta central é simples e desconfortável. Estabilizar a população, ou seja, atingir uma média global de dois filhos por mulher, exigiria retroceder em direitos das mulheres? Forçar parto? Restringir aborto? Empurrar mulher de volta pra casa? Os autores respondem com firmeza absoluta: não. E mais ainda. A única forma viável de estabilizar é justamente o oposto. É tornando paternidade e maternidade mais fáceis, mais compartilhadas, mais justas. Qualquer caminho que envolva coerção, segundo os dados, simplesmente não funciona. Vamos ver.

O capítulo abre com uma história pessoal. Junho de dois mil e vinte e dois, Texas. April descobre que está grávida. Ela tem quarenta anos, está tentando o segundo filho com o marido há dois anos. Nesse mesmo mês, a Suprema Corte americana derrubou o Roe versus Wade, e o Texas pôs em vigor uma das leis de aborto mais restritivas do país. Aos quarenta, chance de aborto espontâneo é alta. April vai ao obstetra implorando por garantia de que se algo der errado ela vai ter cuidado médico adequado. O médico não consegue prometer. Ela chora. O marido senta na cadeira do canto, mãos enfiadas entre os joelhos pra esconder o tremor.

Algumas semanas depois ela aborta espontaneamente, fora do Texas, com medo de não conseguir tratamento se ficasse. Depois disso, o casal decidiu parar de tentar. A nova lei texana, na prática, encerrou a tentativa do segundo filho.

E aí Spears e Geruso revelam: April é casada com o Mike. O Mike é Michael Geruso, um dos autores do livro. A cena no consultório era ele. Eles não foram garimpar essa história. Ela aconteceu com eles. E é a porta de entrada do capítulo porque ilustra um ponto muito direto: políticas que restringem liberdade reprodutiva não aumentam o número de bebês. Pelo contrário, em casos como esse, diminuem.

A partir daí o capítulo desmonta cinco assumições erradas. Vou te apresentar uma de cada vez.

Primeira assumição errada: pra atingir média dois, todo mundo precisa ter dois. Falso. Média é média. Os autores comparam Canadá e Tchéquia. Mulheres nascidas em mil novecentos e setenta e três nos dois países tiveram em média um vírgula oito filho. Mas o jeito de chegar nessa média foi diferente. Na Tchéquia, mais da metade teve exatamente dois. No Canadá, menos de quarenta por cento. No Canadá, mais mulheres tiveram três ou mais, e quase o dobro tiveram zero. A média foi a mesma. Família grande, família média, família sem filho, todas convivem dentro da mesma média global de dois. Estabilização não é uniformidade. Não é coação. É um agregado.

Segunda assumição errada: pra natalidade subir, progresso feminino tem que parar ou retroceder. Falso. E aqui os autores trazem um caso histórico curioso. Entre mil novecentos e setenta e cinco e dois mil e dez, ou seja, trinta e cinco anos, a taxa de fertilidade americana ficou praticamente estável, oscilando perto de um vírgula nove. Foi uma anomalia. Na Europa caiu de dois vírgula zero sete pra um vírgula seis. No Japão, de um vírgula nove pra um vírgula quatro. Nos Estados Unidos, ficou parada.

E foram justamente esses trinta e cinco anos os de maior progresso para a mulher americana. A diferença salarial caiu de sessenta por cento pra setenta e cinco por cento. Mais mulheres no ensino superior, mais mulheres em cargos elevados, primeira mulher na Suprema Corte foi em mil novecentos e oitenta e um. Equal Credit Opportunity Act em setenta e quatro. Casamento igualitário em dois mil e quinze. Tudo isso enquanto a fertilidade ficou parada em um vírgula nove. Ou seja, é possível, é factível, que progresso feminino e fertilidade próxima de dois coexistam.

Terceira assumição errada: países mais igualitários têm fertilidade menor. Falso. Olha o que os dados de países da OCDE mostram. Espanha, Itália e Japão têm a mesma fertilidade de um vírgula três. Mas o pay gap é três por cento na Itália, sete na Espanha, vinte e um no Japão. Sem padrão. A Irlanda, com fertilidade de um vírgula oito, uma das maiores da Europa, tem pay gap de dois por cento, dos menores. A correlação entre pay gap e fertilidade, no universo dos países ricos, é zero. Tirando a Coréia do Sul, que é um outlier extremo, variação no pay gap explica menos de zero vírgula dois por cento da variação de fertilidade. Tipo dois milésimos.

E o caso da Coréia do Sul, esse outlier, reforça o argumento dos autores na direção oposta. A Coréia do Sul é o país desenvolvido com o maior pay gap, e tem a menor fertilidade do mundo. Por quê? Narae Park, economista coreana que trabalha com Spears, explica: a economia coreana cresceu rápido, mas as normas culturais não. Mulher coreana é altamente escolarizada, mas se tiver filho vira a cuidadora principal, perde carreira. Então ela hesita em casar e ter filho. Pra subir a fertilidade na Coréia, precisaria diminuir o pay gap, não aumentar. A igualdade não é inimiga da fertilidade. É condição.

Suécia e Dinamarca, países no topo do Índice de Igualdade de Gênero da União Europeia, têm fertilidade mais alta que Grécia e Hungria, que estão no fim do ranking. Sociedades mais igualitárias parecem ter mais filhos, não menos. Não é prova final, é indicação.

Quarta assumição errada, e essa é central: maternidade é coisa biológica de mulher, não tem o que fazer. Falso. Spears e Geruso citam a economista Claudia Goldin, ganhadora do Nobel em dois mil e vinte e três, que estudou o pay gap em carreiras de alta exigência. Goldin chama de empregos gulosos, greedy jobs. São os que pagam muito mas exigem disponibilidade total. Num casal sem filhos, os dois podem aceitar. Num casal com filhos, geralmente só um. E historicamente esse um foi o homem.

E aí entra a estatística do U.S. Bureau of Labor de dois mil e vinte e um. Em dia útil, vinte e um por cento dos homens fazem alguma tarefa doméstica, contra quarenta e nove por cento das mulheres. Mulheres gastam duas vírgula sete horas em casa quando fazem, homens duas vírgula uma. Em família com filho menor de seis anos, mulher gasta uma vírgula dois hora cuidando ativamente da criança, alimentando, vestindo, trocando. Homem, zero vírgula cinco. E pra eventos escolares em dias de semana, a tabela do Bureau tem uma nota de rodapé que diz: estimativa aproximadamente zero. Pais homens em dias de semana não vão. Zero.

A solução proposta pelos autores não é exigir mais das mulheres. É exigir mais dos homens. Eles escrevem uma frase ótima: leva mais que nove meses pra fazer uma pessoa. Leva anos. Tem muito espaço pros homens entrarem. Limpar peça de bombinha de leite, levantar às três da manhã, cozinhar à noite, lavar roupa, levar pra natação, levar pra festa de aniversário. Spears e Geruso dizem: se você está lendo isso daqui a cem anos, esperamos que esse parágrafo soe estranho, anacrônico. Tipo, espera, antigamente os pais não cuidavam dos filhos?

E não para nos homens. Eles dizem: estabilização populacional é interesse coletivo. Logo, o ônus deveria ser coletivo. Quem não tem filho hoje deveria carregar parte do peso de quem está criando filho, via impostos, via tempo, via comunidade. Eles voltam nisso na Parte quatro.

Quinta assumição: gravidez é necessariamente sofrimento, então tem mulher que jamais vai querer. Aqui os autores apontam algo desconfortável. Existem tratamentos pra náusea severa, existem pesquisas pra reduzir os pioneiros, mas o financiamento pra saúde da mulher historicamente foi menor que pra saúde do homem. Até mil novecentos e noventa e três, era normal que ensaios clínicos excluíssem mulheres em idade fértil. Tem espaço pra inovação médica que tornaria gravidez menos miserável. Mas é escolha social investir. Não tem investido proporcionalmente.

Fechando o capítulo, eles voltam à pergunta da colega. Quem vai ter os bebês? Mas reformulam. Quem vai criar os filhos? Quem vai sustentar quem cria? Quem vai trabalhar pra mudar economia, ciência e cultura pra que homens e mulheres consigam, querendo, ter média de dois filhos numa sociedade livre, próspera e justa? Resposta: todo mundo.

A síntese desse capítulo é potente. Não tem dilema inescapável entre vida boa pra mulher e estabilização. As duas coisas convivem. Mas exigem trabalho. E exigem que o trabalho seja redistribuído, não esmagado em cima de quem já carrega.

Próximo episódio, capítulo cinco. Adicionar vidas a um mundo imperfeito. Os autores enfrentam aquele argumento profundo, sentimental: por que trazer mais gente a um mundo onde tantos já sofrem? Vai ter ética, vai ter Ehrlich de novo, vai ter discussão sobre antinatalismo. Te espero lá.""",
    },
    {
        "id": "06_mundo_imperfeito",
        "title": "Cap. 5 — Adicionar novas vidas a um mundo imperfeito",
        "text": """Episódio seis. Adicionar novas vidas a um mundo imperfeito.

No episódio anterior, falamos sobre direitos reprodutivos e a tese de que estabilização populacional é compatível com mais igualdade, não menos. Hoje o livro entra no terceiro grande argumento contra mais pessoas. E talvez seja o mais profundo emocionalmente. Por que trazer uma criança a um mundo cheio de sofrimento? Por que adicionar mais vidas a um planeta que talvez não tenha o suficiente pra todo mundo?

O capítulo cinco se chama exatamente isso: Adicionar novas vidas a um mundo imperfeito.

Spears e Geruso abrem com uma cena de café da manhã. Mike Geruso, o autor, está com a família. O pai dele, Bob, nascido em mil novecentos e quarenta e três, faz uma pergunta direta, daquelas que pais fazem quando não querem nem podem fingir educadinhos. Mike, essa gente que defende aumentar fertilidade, está pensando o quê? E a fome, e a falta de comida? Não devíamos estar preocupados em alimentar quem já existe?

O pai do Mike, Bob, cresceu pobre em Woonsocket, Rhode Island, dividindo cama de solteiro com dois irmãos, vestindo herança de roupa, bebendo leite em pó de doação do governo. Não esquece que necessidades básicas não estão garantidas. Mike prometeu pensar. O capítulo é a resposta.

E a resposta começa enfrentando o pai espiritual do argumento, que é, de novo, o Paul Ehrlich. A Bomba Populacional, mil novecentos e sessenta e oito, abre dizendo: a batalha pra alimentar a humanidade acabou. Nos anos setenta e oitenta, centenas de milhões de pessoas vão morrer de fome, e nada pode impedir isso. Foi um chamado pra controle populacional radical. Em sua versão moderna, viraram preocupações sobre escassez de cobre pra carros elétricos, lítio pra baterias, e por aí vai.

Spears e Geruso mostram que Ehrlich errou em tudo. Centenas de milhões não morreram de fome nos anos setenta e oitenta. Pelo contrário. Entre mil novecentos e sessenta e um e dois mil e vinte, a população mundial mais que dobrou, de três vírgula um bilhões pra sete vírgula oito bilhões. E em todos os continentes, sem exceção, a quantidade de calorias disponíveis por pessoa cresceu. O mundo produz hoje cerca de cinquenta por cento mais comida por pessoa do que em mil novecentos e sessenta e um. O demógrafo David Lam fez uma frase irônica num discurso da Associação Populacional Americana em dois mil e onze: nós sobrevivemos à bomba populacional.

Quando Mike mostrou o gráfico pro pai dele, Bob mudou de ideia.

O que existe, sim, é fome localizada. Mas fome moderna não é problema biológico de calorias. É problema político. O Nobel de Economia Amartya Sen é citado: nenhuma democracia jamais teve fome em massa. Não porque democracia previne seca ou enchente, mas porque governo democrático presta contas. Tendo comida no mundo, e sempre tem, governo democrático entrega. A organização Oxfam, que combate fome, diz a mesma coisa em outras palavras: fome não é fenômeno natural, é falha política catastrófica.

Tem outra forma de ver progresso nutricional, e essa é bonita. A altura média das pessoas. Em países pobres, mais velhos são mais baixos. Em países ricos, mais altos. Não é genética. É a comida da infância. Quando criança come bem e não pega muita infecção, ela cresce até seu potencial genético. Quando não, fica abaixo. O fato de cemitérios europeus antigos estarem cheios de esqueletos pequenos diz a mesma coisa: a comida lá era ruim.

Em quinze anos, a criança indiana média de cinco anos cresceu um vírgula sete centímetros de altura. Não é dieta, não é genética, é alimentação melhor, ar mais limpo, menos doença infantil. Visível na fita métrica. Spears, que mora em Uttar Pradesh, anda por aí em hospital com fita métrica medindo. Os dados melhoraram drasticamente.

Aí o capítulo enfrenta o medo dos recursos finitos. Em mil novecentos e oitenta, o economista Julian Simon fez uma aposta pública com Ehrlich. Simon defendeu que recursos naturais ficariam mais baratos com o tempo, porque a engenhosidade humana é o recurso renovável definitivo. Ofereceu pra Ehrlich escolher cinco commodities. Ehrlich escolheu cromo, cobre, níquel, estanho e tungstênio. Mil dólares na cesta em mil novecentos e oitenta. Dez anos depois, a cesta ajustada por inflação valeu menos. Ehrlich pagou quinhentos e setenta e seis dólares pra Simon. A cesta inteira ficou cinquenta por cento mais barata. Por quê? Engenhosidade. Material alternativo, mineração mais eficiente, e em alguns casos, substitutos: lata de leite hoje em dia é feita de ferro, não estanho. Silício, abundante na crosta, virou processador de computador, virou semicondutor, virou smartphone, virou tudo.

Tem uma anedota linda. Mike, com nove anos, tinha pavor de que o tungstênio acabasse. A professora dele falou que o filamento da lâmpada elétrica precisava de tungstênio, e que o tungstênio estava acabando. Mike, criança, deitava na cama olhando pra abajur com medo. Hoje, ninguém usa lâmpada de tungstênio. Foi substituída por LED. Problema sumiu sem ser problema.

A virada do capítulo entra com uma cena emocional. A escritora Britt Wray descreveu o medo de ter filho num mundo dominado por humanos gananciosos rumando pra zonas mortas ecológicas. Spears e Geruso, eles mesmos pais, reconhecem essa angústia. Dean Spears e a esposa, Diane, tiveram dificuldade pra decidir ter filho. Vários abortos espontâneos, FIV que falhou, finalmente conseguiram numa ultrassonografia num porão de hospital em Lucknow. E mesmo lá, sabendo o que sabiam sobre o mundo, sobre Uttar Pradesh, sobre poluição, sobre mortalidade infantil, eles decidiram seguir. Toda criança é uma aposta. Eles acharam que a chance de uma vida boa era maior que a chance de uma ruim.

E aí vem dados gerais. Expectativa de vida global em mil e oitocentos: menos de trinta anos. Em mil novecentos e sessenta e oito, quando Ehrlich publicou A Bomba: cinquenta e sete. Hoje: setenta e três. Em mil novecentos e setenta, a expectativa de vida do americano negro era muito menor que a do branco. Nos anos dois mil, a do branco subiu, mas a do negro subiu duas vezes mais. Não fechou, mas o gap diminuiu. Globalmente, a humanidade ganha em torno de três vírgula sete meses de vida por ano desde Ehrlich.

E mais. Hoje existem remédios pra HIV, pra diabetes, pra polio, pra ansiedade, pra cegueira congênita, pra náusea de gravidez. Em mil novecentos e sessenta e oito, naproxeno, o Aleve, não existia. Hoje custa quatro centavos de dólar o comprimido. Pra alguém com salário americano médio de trinta e cinco dólares por hora, cinco segundos de trabalho pagam o comprimido que tira a dor sem efeito colateral grave. Os reis do passado não tinham isso. Você tem.

E quanto a piora recente, queda da expectativa de vida americana entre dois mil e vinte e dois mil e vinte e um por causa de COVID, mortes por desespero, opioides? Spears e Geruso mostram um gráfico. A queda foi real, mas voltou o nível pra mais ou menos o que era em mil novecentos e noventa. Você diria que vida em mil novecentos e noventa nos Estados Unidos não valia ser vivida? Em mil novecentos e setenta? Em mil novecentos e quarenta e três, quando Bob nasceu pobre em Rhode Island? Bob, hoje, está feliz com a vida dele. Leite em pó e tudo.

A conclusão moral do capítulo é nuanceada. Os autores não negam que o mundo tem problemas urgentes. Mudança climática, fogo, desigualdade, doenças. Tudo verdadeiro. Tudo que a humanidade precisa atacar. Mas a tese deles é que esses problemas não tornam vidas humanas futuras não dignas de serem vividas. Mais sofrimento, sim, em alguns lugares e tempos. Vida no geral pior que vida sem mudança climática, sim. Mas vida ainda melhor que a dos bisavôs.

E, aqui, eles encontram um ponto comum até com quem pensa diferente. Eles dizem o seguinte: nós concordamos com quem diz que não se deve criar uma vida que será ruim. Concordamos. A pergunta é só sobre os fatos. Se vidas como a sua, hoje, valem ser vividas, então a vasta maioria das vidas futuras também vai valer.

E essa concordância vai abrir o gancho enorme do livro, que é a ética populacional. Se vidas boas adicionadas ao mundo são uma coisa boa, então deixar de adicionar muitas vidas boas é, de algum jeito, uma perda. Isso vai ser desenvolvido no capítulo oito. Por enquanto, a mensagem é: não tenha medo de gerar um filho num mundo imperfeito. O mundo sempre foi imperfeito. As pessoas dele viveram, amaram, criaram, melhoraram tudo. E o teu filho, estatisticamente, vai viver melhor do que você viveu.

Próximo episódio, capítulo seis. Aí começa a Parte três do livro, o argumento a favor das pessoas. O capítulo se chama Progress Comes from People. Por que mais gente, na economia inteira, gera mais ideias, mais saltos, mais inovação. E por que despovoamento freia esse motor. Te espero lá.""",
    },
    {
        "id": "07_progresso_vem_de_pessoas",
        "title": "Cap. 6 — Progresso vem de pessoas",
        "text": """Episódio sete. Progresso vem de pessoas.

Os três episódios anteriores destruíram, um por um, os argumentos contra mais pessoas. Ambiente. Direitos. Sofrimento. Agora começa a Parte três do livro, e aqui Spears e Geruso saem da defesa e partem pro ataque. O argumento positivo. Por que mais gente é melhor. Capítulo seis: Progresso vem de pessoas.

A ideia central do capítulo cabe numa frase: ideias são geradas por pessoas, ideias não se gastam quando são usadas, e por isso mais pessoas significam mais progresso pra todo mundo. Mas pra você sentir o peso disso, vamos por etapas, começando, mais uma vez, pelo hospital indiano.

Preeti, lembra dela, mãe nova em Uttar Pradesh, está há cinco semanas no hospital. A bebê dela nasceu com mil e setenta gramas. Hoje está com mil trezentos e quinze gramas. Quase um quilo de ganho. A enfermeira Seema e a gerente Kavitha decidem que a bebê pode ir pra casa. Ela está se alimentando, Preeti aprendeu a técnica. Hospital tem risco de infecção e a cama precisa de outra mãe.

A técnica que Preeti aprendeu se chama Cuidado Mãe Canguru, ou KMC na sigla em inglês. Não é alta tecnologia. É um processo. Manter o bebê em contato pele a pele constante com a mãe, inclinado, amarrado pra não escorregar, amamentando livremente. Esse processo foi inventado em Bogotá, Colômbia, em mil novecentos e setenta e oito. Os pediatras de lá não tinham incubadoras. Inventaram alternativa. Em dois mil e vinte e um, o New England Journal of Medicine publicou um estudo grande, ensaio randomizado em Gana, Índia, Malawi, Nigéria e Tanzânia. Resultado: bebês prematuros sob cuidado canguru tinham vinte e cinco por cento menos chance de morrer no primeiro mês comparado com os que iam pra incubadora. Vinte e cinco por cento. Salva milhares de vida todo ano.

E qui está o ponto que o capítulo quer instalar na sua cabeça. KMC é uma tecnologia. Não tecnologia de chip, tecnologia de processo. Foi inventada uma vez em Bogotá. Pode ser copiada em qualquer hospital do planeta, infinitamente, sem se gastar. A Organização Mundial da Saúde publica o manual gratuitamente. A Clínica Cleveland também. Cada hospital novo que adota KMC não tira KMC de ninguém. Não há rivalidade no uso. Isso, no jargão econômico, se chama inovação não rival.

E aqui o livro me apresenta um economista importante, o Paul Romer, que ganhou o Nobel em dois mil e dezoito justamente por entender isso. Romer fez o seguinte: até os anos oitenta, os economistas explicavam crescimento principalmente pelo acúmulo de capital físico, máquinas, fábricas, estradas. Mas isso não bastava. Tinha que haver mudança tecnológica. E mudança tecnológica vinha de onde? Os modelos antigos não respondiam.

Romer respondeu. Vem de gente. Pessoas inventam. Pessoas refinam. Pessoas testam. E uma vez inventada, a ideia pode ser usada por todo mundo, sem fim. No discurso do Nobel, em Estocolmo, Romer disse uma frase elegante. Ideias podem ser compartilhadas. Não no sentido de revezamento, mas no sentido de que todo mundo pode usar o teorema de Pitágoras ao mesmo tempo.

Outro exemplo lindo de inovação não rival. Sal de reidratação oral. Misturinha de sal, açúcar e água, dissolvida em proporção certa. Inventado num hospital de cólera em Bangladesh nos anos sessenta. Salva milhões de crianças por ano. Em Uttar Pradesh, é água de geladeira misturada em casa. Em Austin, Texas, é Pedialyte sabor uva em garrafinha. Mesma receita, formas diferentes, salvando bebê em qualquer lugar do mundo. Antes dos anos sessenta, essa receita simplesmente não existia.

A boa notícia: ideias não se gastam. A má notícia: ideias precisam ser descobertas por alguém. Não chovem do céu.

E aqui vem o argumento decisivo do capítulo. Chad Jones, economista de Stanford, foi um dos primeiros a perguntar: o que acontece com o motor da inovação quando a população encolhe? Resposta dele: o motor desacelera. Não tem como contornar. Menos pessoas, menos cientistas, menos engenheiros, menos professores, menos pesquisadores, menos curiosos numa garagem, e portanto menos ideias.

Spears e Geruso citam o economista Michael Peters da Yale, que escreveu numa revista de prestígio o seguinte: virtualmente todas as teorias de crescimento econômico preveem uma relação positiva entre tamanho da população e produtividade.

Tem uma objeção comum, que é: tudo bem, mas vamos investir mais em educação pra produzir mais cientistas per capita. Spears e Geruso topam. Concordam que investir em educação é ótimo. Mas mostram que isso é insuficiente. Olha o número. Se cinquenta por cento de uma população de um bilhão fizer ciência, são quinhentos milhões. Se dez por cento de dez bilhões fizer, é um bilhão. Pra inovação total, o que conta é número absoluto, não fração. E é número absoluto que despovoamento destrói.

E tem mais. Romer mostra que progresso é cumulativo. Ideias se combinam com ideias anteriores. O capítulo dá um exemplo curioso. O beisebol passou a ter relógio de arremesso em dois mil e vinte e três. Por que? Porque o basquete inventou o shot clock em mil novecentos e cinquenta e quatro. E o basquete só existe porque a regra dos Knickerbockers do beisebol de mil oitocentos e quarenta e cinco existia. Inovação se empilha. Se você perde um andar, perde tudo que vinha em cima.

Pra fechar essa intuição, os autores trazem uma história deliciosa sobre luz. Iluminação artificial. O economista William Nordhaus fez um trabalho fascinante em mil novecentos e noventa e quatro. Ele queriacomparar GDP do passado com o presente, mas como? Apple do passado é diferente da apple de hoje. Não tem como. Mas tem uma coisa que dá pra medir igualzinho ao longo da história: lúmens. Quantos lúmens de luz uma hora de trabalho compra?

Nordhaus reuniu lenha, gordura animal, óleo de gergelim, e tocou fogo, literalmente. Mediu com luxímetro. Resultado: na Babilônia, mil setecentos e cinquenta antes de Cristo, uma hora de trabalho num pobre comprava uns poucos lúmens de óleo de gergelim. Em Paris medieval, candelabro caro. Século dezenove, lampião a querosene. Mil oitocentos e setenta e nove, Edison e a lâmpada incandescente. Anos oitenta, fluorescente. Hoje, LED.

Quantas vezes mais luz uma hora de trabalho compra hoje em relação à Babilônia? Trezentas e cinquenta mil vezes mais. Trezentas e cinquenta mil. E olha o formato do gráfico de Nordhaus. Estagnação longa, depois explosão. Que coincidência. O formato é o mesmo do gráfico da população humana. Estagnação longa, explosão recente.

Spears e Geruso não dizem que é prova causal. Não dá pra rodar ensaio randomizado em civilizações. Mas seria coincidência absurda se essas duas curvas tivessem o mesmo formato e a população não tivesse papel nenhum nisso.

E tem uma observação importante. Progresso não vem de gênio solitário. Edison não inventou a lâmpada sozinho. Sawyer e Man patentearam coisa parecida. Swan na Inglaterra também. Foi um esforço coletivo, cumulativo, de muita gente. O argumento do livro não é precisar de mais gente porque toda criança é um bilhete de loteria pra próximo Einstein. É que mesmo num mundo onde todas as pessoas fossem igualmente medianas em criatividade, mais pessoas ainda gerariam mais ideias e mais combinações. A força está na quantidade, não na excepcionalidade.

E quanto à inteligência artificial? E se a IA passar a inventar tudo, então não precisa de gente? Spears e Geruso são honestos. Eles não descartam. Mas dizem: seria irresponsável apostar tudo nisso. A IA pode ajudar muito, pode complementar humanos, pode até superar em algumas tarefas. Mas afirmar com certeza que ela substituirá todo pensamento humano em breve é tão arriscado quanto afirmar com certeza que não. Planeje pros dois cenários. E os dois precisam de gente.

A síntese desse capítulo é firme. Toda apple do supermercado hoje, toda pílula, toda vacina, todo LED, toda receita de Pedialyte, é resultado de descobertas que alguém precisou fazer. E o motor de fazer descoberta roda em uma matéria-prima só: pessoas pensando, testando, errando, trocando, ensinando.

Se a humanidade despovoa, esse motor não morre, mas perde combustível. Cada geração que vem menor tem menos chance de ter o time que vai resolver o próximo problema. Câncer pode demorar mais. A geotermia que substituiria o carvão pode demorar mais. A nova terapia gênica que cura cegueira pode demorar mais. Não é abstrato. São anos de vida, anos de sofrimento, vidas inteiras.

Mais gente vivendo bem é mais ideias acumuladas pra todos. Esse é o argumento positivo do livro. Não é sobre crescer pra ser maior. É sobre não desligar a esteira da invenção justo agora.

Próximo episódio, capítulo sete. Desviando do asteroide e outros benefícios de mais pessoas. Aí o livro entra num campo curioso, risco existencial. Como mais gente protege a espécie inteira de eventos catastróficos, como asteroides, vulcões, pandemias. Te espero lá.""",
    },
    {
        "id": "08_desviando_asteroide",
        "title": "Cap. 7 — Desviando do asteroide",
        "text": """Episódio oito. Desviando do asteroide, e outros benefícios de mais pessoas.

No episódio passado, capítulo seis, Spears e Geruso mostraram que progresso vem de pessoas, porque ideias não se gastam quando usadas. Hoje, capítulo sete. Mesma tese, mas por outro ângulo. O argumento é sobre custos fixos. Pra fazer uma coisa boa existir, alguém tem que pagar um custo fixo. Quanto mais gente divide esse custo, mais coisa boa cabe. E quando a humanidade encolhe, vários quesitos importantes deixam de existir, ou demoram mais. Vou te contar.

Eles começam com Preeti em casa. A bebê está bem, mas o time do hospital ainda visita e liga por semanas. Tudo isso funciona. Por quê? Porque Uttar Pradesh tem muita gente, e isso permite escala. Antes, vinte anos atrás, só um quinto dos partos em Uttar Pradesh era em hospital. Hoje, mais de quatro quintos. O hospital onde Seema trabalha vê trinta partos por dia, em média. E desses trinta, um ou dois precisam do programa Mãe Canguru. Sem essa massa de partos, não existiria uma ala dedicada. Não daria pra contratar enfermeira especializada. Não daria pra Dean financiar como caso eficiente. Não daria pra existir.

Isso é custo fixo. Custo fixo é o custo de fazer alguma coisa existir, antes de fazer qualquer quantidade dela. O salário da Seema, a sala, o software de prontuário. Tudo isso existe se tiver dois bebês ou se tiver duzentos. Quando tem dois, o custo per capita é absurdo. Quando tem duzentos, o custo por bebê salvo cai pra trezentos dólares. E é por isso que vale a pena fazer.

A intuição central do capítulo é essa. Sua vizinha não está comendo a fatia da sua torta. Sua vizinha é a razão de alguém estar assando a torta. Você não tem ramen autêntico no seu bairro porque você quer. Você tem porque tem gente o suficiente querendo na mesma região pra justificar o restaurante existir. Sem essa massa de demanda, o restaurante fecha ou nunca abre.

A partir disso, o capítulo desfila uma série de exemplos. Cinema em alemão é muito menor do que cinema em inglês. Por quê? Mercado menor. Vacina contra VRS, vírus respiratório, existe. Vacina contra varíola da foca, não existe. Por quê? Milhões pegam VRS, só algumas dezenas de tratadores de foca pegam varíola da foca. Custo fixo de pesquisa exige escala de mercado.

Cidades são um microcosmo ótimo pra essa ideia. Casa em Austin custa mais que o dobro do que em Tulsa, Oklahoma, cidade onde Dean cresceu. Por quê pagar mais? Pra estar perto de mais gente. Em mil novecentos e sessenta e oito, trinta e seis por cento dos humanos viviam em cidades. Hoje, cinquenta e sete por cento. A urbanização dobrou, e padrão de vida triplicou. Coincidência? Não.

Pra você sentir a interação entre cidade, custo fixo e câncer, eles contam uma história pessoal. Dean voltou pra Tulsa quando o pai e depois a mãe tiveram câncer. Sentou na sala de infusão e conversou com outras famílias que vinham de cidades pequenas do Oklahoma, Jenks, Skiatook, Bristow, Nowata. Esses lugares pequenos não conseguem manter um centro de câncer. Tulsa tem gente suficiente. Os pequenos não. Então quem mora longe viaja. Esse é custo fixo de tratamento médico, e densidade populacional faz a diferença direta no acesso ao tratamento.

E aí o capítulo dá um pulo provocativo. E o seu celular?

Inventar o próximo smartphone custa centenas de milhões de dólares. Fabricar um custa algumas centenas. Por que o smartphone do bilionário é o mesmo seu? Talvez tenha capinha de ouro, mas por dentro é igual. Por quê? Porque mesmo o bilionário, pagando um milhão pelo aparelho, não pagaria sozinho o custo fixo de desenvolvimento. Só dá pra desenvolver porque a empresa vai vender cem milhões de unidades. E olha o ponto. Existe um smartphone barato, robusto, desenvolvido pra fazendeiros e comerciantes pobres na África e no Sul Asiático. Por quê? Porque cinquenta milhões de unidades a cinquenta dólares pagam o custo fixo. Bilionários sozinhos não pagam. Pobres em grupo pagam. Os pobres do mundo são o motor de inovação que bilionário sozinho não consegue ser.

E essa é a tese que mais gostei do capítulo: o aparelho que tá na sua mão existe porque a Terra tem muita gente. Reduz a humanidade pra um bilhão de pessoas em vez de oito, e várias coisas que você tem hoje deixam de existir. Um leitor do New York Times escreveu pros autores: bom, um mundo com um centésimo da população não teria iPhones, Airbus, jogos de videogame com produção pesada. Não teria mercado pra produzir. Spears e Geruso concordam. E acrescentam: provavelmente também não teria computadores portáteis, editoras de livro, medicamentos pra câncer raro.

Pra deixar essa ideia palpável, eles contam a história da vacina de mRNA contra COVID. mRNA é abreviação de ácido ribonucleico mensageiro. A tecnologia existia há setenta anos, mas só conseguiu ser usada em vacina em dois mil e vinte. Por trás, tem a Kátalin Karikó, bioquímica, demitida da Universidade da Pensilvânia por não conseguir financiamento de pesquisa. Insistiu por décadas. Em dois mil e cinco, ela e o imunologista Drew Weissman descobriram como modificar quimicamente o mRNA pra não ser destruído pelo sistema imune. Em mil novecentos e oitenta e sete, um estudante chamado Robert Malone testou bolinhas de gordura pra entregar mRNA em embriões de sapo. Em dois mil e um, isso virou nanopartícula lipídica. Em dois mil e dezoito, primeiro remédio aprovado com essa tecnologia. Em dois mil e vinte, vacina contra COVID. Karikó e Weissman ganharam o Nobel de Medicina em dois mil e vinte e três. Mas eles dizem o tempo todo: o crédito é coletivo. O biofísico Pieter Cullis, que liderou as nanopartículas, fala em milhares de pessoas. Milhares.

Pra produzir um único frasco daquela vacina, foi preciso uma rede gigante. Cientistas líderes. Pós-docs. Equipe logística de ensaio clínico. Revisores de pares. Burocratas do NIH avaliando pedidos de bolsa. Engenheiros que fabricaram bancadas, microscópios, centrífugas. Programadores que fizeram o software estatístico. Todos contemporâneos. Spears e Geruso perguntam: num mundo com um bilhão de pessoas em vez de oito, essa vacina teria existido em tempo de salvar quem ela salvou? Resposta deles: provavelmente não. A receita ficaria meio escrita. Os nichos não estariam preenchidos.

Tem uma analogia simples. Se quatro pessoas empurram um móvel pesado juntas, ele anda. Se cada uma empurra sozinha por sua vez, fica parado. Tarefas complexas não escalam linearmente como xícara de açúcar em receita de biscoito. Algumas só funcionam com massa crítica de gente especializada.

E aqui o capítulo entra no momento que dá título ao episódio. Asteroides. Não estou brincando.

Spears e Geruso querem te mostrar que existe uma classe inteira de riscos existenciais que segue exatamente a mesma lógica de custo fixo. Asteroide vindo na direção da Terra, supervulcão continental, pandemia tão contagiosa quanto sarampo e tão letal quanto raiva. Esses riscos custam o mesmo pra resolver, independente da população. Se desviar o asteroide custa dez trilhões de dólares, custa dez trilhões pra matar dez bilhões ou pra matar dez milhões.

Aí eles fazem uma conta simples. Dois futuros. Em ambos, a renda per capita é setenta mil dólares, mais ou menos a americana atual. Um futuro com dez bilhões de pessoas. O outro com dez milhões. Asteroide vem nos dois. Custo de desvio: dez trilhões.

No futuro de dez bilhões, esse custo é um vírgula cinco por cento da economia global. Apertado, mas viável. Aumento de imposto pesado por alguns anos. Algumas indústrias suspensas. Mas dá pra fazer.

No futuro de dez milhões de pessoas, mesmo custo absoluto. Mas a economia inteira anual é setecentos bilhões. Você precisaria gastar mil e quinhentos por cento da economia global em um único projeto. Impossível. Mesmo se todo mundo parasse tudo e dedicasse cem por cento dos esforços, faltariam noventa e três por cento dos recursos. Asteroide passa. Humanidade morre.

A população maior tem mais chance de sobreviver a riscos existenciais com custo fixo. Não só asteroide. Também próxima pandemia, próximo supervulcão, próxima tecnologia perigosa que precise ser contida.

E aplicando essa lógica ao clima. Lembra que no capítulo três os autores disseram que despovoamento não ajuda contra mudança climática porque é lento? Beleza. Mas tem um problema climático que vem depois. Quando a humanidade atingir emissão líquida zero, o CO dois acumulado na atmosfera ainda vai estar lá. Hoje está em quatrocentos e vinte e cinco partes por milhão. Antes da revolução industrial, era duzentos e oitenta. A diferença, cento e quarenta e cinco partes por milhão, é o estoque acumulado. Esse estoque vai ter que ser removido. Reflorestamento, sequestro geológico, novas tecnologias. Tudo isso vai custar uma fortuna fixa. E o tamanho dessa fortuna é o mesmo independente da população.

Conclusão dos autores: uma humanidade maior, mais rica e mais populosa, paga o sequestro de carbono mais rápido e com menos sacrifício individual. Despovoamento corta exatamente os recursos que a próxima geração vai precisar pra limpar o céu que herdou.

A última frase do capítulo eu acho perfeita. Não dá pra saber com certeza o que uma economia complexa moderna conseguiria produzir se houvesse poucas pessoas pra preencher seus nichos. A especialização nichada que faz a sua vida boa é recente demais. Ninguém deveria fingir que sabe o que aconteceria sem ela.

Próximo episódio, capítulo oito. Mais bom é melhor. Esse é o capítulo mais filosófico do livro. Spears e Geruso encaram a ética populacional de frente. Vidas adicionais valem? Mais felicidade no mundo é mais bem? Vou tentar te contar com cuidado. Te espero lá.""",
    },
    {
        "id": "09_mais_bom_e_melhor",
        "title": "Cap. 8 — Mais bom é melhor",
        "text": """Episódio nove. Mais bom é melhor.

Esse é o coração filosófico do livro. Capítulo oito. Os autores se chamam economistas, mas aqui eles fazem filosofia moral. E a tese é simples de enunciar e difícil de digerir. Vou tentar te conduzir com cuidado.

A tese é: se uma vida boa for vivida, isso conta como uma coisa boa. E se uma vida que seria boa não chega a existir, isso conta, em algum grau, como uma perda. Mais bem é melhor. Por isso o capítulo se chama More Good Is Better.

Antes de pular pra teste, eles fundamentam culturalmente. Em todas as tradições religiosas e seculares mais respeitadas, vida humana é um valor. Carl Sagan, agnóstico, ateu pra muitos efeitos, dizia que somos raros e preciosos porque estamos vivos. Ursula Le Guin escreveu que a gente não vive pra morrer, vive pra viver. O Alcorão pergunta como você pode descrer de Deus se Ele te deu vida quando você era nada. Thich Nhat Hanh dizia que o maior milagre é estar vivo. O Velho Testamento manda crescer e multiplicar. Mais ou menos toda tradição concorda que estar vivo é melhor que não estar.

Agora a parte filosófica.

A pergunta que o livro coloca, e que pouca gente para pra fazer, é: faz diferença pro mundo se uma pessoa adicional, com vida boa, chega a existir? Se sim, despovoamento não é só um problema econômico. É uma perda ética. Bilhões de vidas potenciais boas nunca aconteceriam.

Aí Spears e Geruso, pra você pensar com clareza, montam um experimento mental que veio do filósofo John Broome. Vou tentar transmitir. Imagina dois futuros possíveis.

Futuro A: é o status quo. Você existe, eu existo, todo mundo que existe está aí. Vidas tem altos e baixos, alguns bem, outros mal.

Futuro B: pega o futuro A e duplica. Cada pessoa do A existe igualzinha no B, com a mesma vida. Mas adicionalmente, no B, existe uma outra metade da humanidade, e essa metade tem vidas excepcionalmente boas. Melhores que qualquer vida no A. Sem ônus pra ninguém. Não desmata mais um centímetro do planeta. Não afeta o salário de ninguém. As pessoas extras simplesmente existem felizes.

Qual é melhor, A ou B?

Spears e Geruso dizem: B, óbvio. Mais vidas boas, sem prejuízo pra ninguém. Não tem como argumentar honestamente o contrário.

E aí eles enfrentam uma posição comum, que eles chamam de só qualidade não importa quantidade. Essa posição diz: o que importa é o nível médio de bem-estar, não o número absoluto de pessoas. Se a média sobe, melhor.

Os autores mostram por que essa posição não funciona. Imagina o futuro C, que é o futuro B mais uma pessoa. Essa pessoa nova tem vida feliz, melhor que a sua e a minha, mas um pouquinho abaixo da média do futuro B, que já era super alta. Pela lógica da média, adicionar essa pessoa felicíssima piora o mundo, porque puxa a média pra baixo.

Isso é absurdo. Adicionar uma vida boa não pode piorar o mundo. Essa pessoa quer viver, ninguém é prejudicado pela existência dela, ninguém ao redor sente menos felicidade por ela existir. Só uma posição absorvida pela média estatística diria que esse acréscimo é ruim. Spears e Geruso dizem: essa posição está errada. Está medindo a coisa errada.

Outro exemplo, agora doloroso. Imagina uma menina nascendo em Uttar Pradesh ano que vem. Pobre pelos padrões americanos. Mas com sorte na família, na saúde, na escola que ela curte, no trabalho como enfermeira que ela acha digno. Casamento arranjado bom, irmãos amigos, vida abaixo da média global por critério de renda, mas que ela mesma, refletindo, não trocaria por não-existência. Pela lógica da média, essa menina deveria não nascer, porque puxa a média pra baixo. Os autores dizem isso é uma conclusão monstruosa. Quem somos nós pra dizer que a vida dessa menina não vale ser vivida, se ela mesma a valoriza?

A correção é a tese do capítulo. Quantidade conta. Cada vida boa adicional é um bem em si.

Tem uma história linda no capítulo, parábola conhecida. Você caminhando na praia depois da tempestade. Tem milhares de estrelas-do-mar encalhadas. Uma pessoa vai pegando uma a uma e jogando de volta no mar. Você diz: mas tem milhares, você não vai fazer diferença. A pessoa joga mais uma e responde: fiz pra essa.

O ponto: quando você compara dois futuros, importa só o que muda entre eles. Se a única diferença é que uma vida boa adicional acontece, e nada mais, então essa diferença é uma melhora. Independente do tamanho global. Independente da média. Faz diferença pra essa.

Outra objeção comum, e os autores levam a sério. Se mais vidas boas é melhor, então não é melhor cada um ter vinte e cinco filhos? Vinte e cinco vidas boas? Resposta: não. Acreditar que algo é bom não é a mesma coisa que acreditar que sempre mais desse bem é melhor a qualquer custo. Alimentar quem tem fome é bom. Mas isso não significa que devo gastar todo meu tempo e dinheiro só nisso, ignorando tudo mais. Vinte e cinco filhos destruiria a saúde da mãe, a sanidade da família, a sociedade. A tese é uma comparação na margem. Adicionar uma vida boa, sem custo pra ninguém, é bom. Não é compromisso com infinito.

E aqui chega o ponto delicado, que os autores antecipam todas vezes. Se mais vidas boas é bom, então não tô justificando obrigar mulher a gerar filho contra a vontade? Não. E essa é provavelmente a parte mais importante do capítulo, então vou desenvolver.

Spears e Geruso usam analogia com doação de rim. A maioria das pessoas saudáveis tem dois rins e vive bem com um. Doar um rim salva uma vida. Doar rim é uma coisa boa. Faz o mundo melhor. Mas você acha que o governo deveria poder me obrigar a doar um rim? Não. Eu sou dono do meu corpo. Bem-estar é importante, mas autonomia do corpo é separada e prevalece. Filósofa Judith Jarvis Thomson é citada nesse argumento. As duas frases não conflitam. Uma: seria melhor se mais rins fossem doados. Duas: ninguém pode ser obrigado a doar.

E pra gravidez é a mesma coisa. As duas frases convivem perfeitamente. Uma: seria melhor se a humanidade não despovoasse. Duas: ninguém pode ser obrigado a engravidar nem proibido de engravidar.

Spears e Geruso fazem ainda outra analogia com a enfermeira Seema. Seema escolheu ser enfermeira, e o trabalho dela salva bebês. Você concorda que mais bebês salvos é bom. Você concorda que o trabalho dela tem valor moral. Mas você acha que o governo deveria poder obrigar Seema a continuar enfermeira contra a vontade dela? Claro que não. As duas coisas se sustentam. O trabalho de enfermagem é bom. Coerção pra exercê-lo é errada. Pra parentalidade, idêntico.

Pra te dar uma ideia do que poderia ser melhorar parentalidade respeitando liberdade, eles contam a história do economista Alvin Roth, Nobel em dois mil e doze. Roth resolveu um problema enorme: doação de rim entre pessoas vivas só funciona se sangue e tecido baterem. Família X quer doar mas não bate. Família Y também. Antes do Roth, paciente esperava na fila pra doador morto. Roth criou algoritmo que faz cadeias de doação. Sua esposa recebe rim de alguém, e você doa o seu pra mãe daquele alguém, que doa pra mais alguém. Cadeias de dezenas de pessoas. Resultado: tempo de espera caiu, transplantes aumentaram, vidas salvas.

Antes do Roth, doação era voluntária. Depois do Roth, doação era voluntária. Mas o sistema redesenhado fez mais bem acontecer dentro da liberdade. O recado do exemplo é: não existe conflito necessário entre respeitar escolha e construir um sistema onde mais coisas boas aconteçam. Pra parentalidade, vale o mesmo. Talvez exista um Al Roth da política familiar esperando pra ser ouvido. A próxima parte do livro vai entrar nisso.

Pra fechar, eles colocam um espelho moral diante do leitor. O escritor médio que escreveu pra eles dizendo eu lembro de mil novecentos e setenta quando éramos metade e era ótimo, ou eu lembro de mil novecentos e cinquenta e cinco e tudo era melhor, esse leitor tem um ponto cego. Ele está vivo. Em mil novecentos e cinquenta e cinco, com a população sendo um terço da atual, talvez ele nem tivesse existido. A pessoa que olha pro despovoamento sem se incomodar é a pessoa que está ela mesma assegurada. Mas as bilhões de pessoas que não existiriam num mundo despovoado não escrevem pra New York Times. Elas simplesmente não estão lá. Você precisa ser advogado delas. Se não, ninguém é.

E pra estender a analogia, ele cita o trabalho da April, esposa do Mike, que é planejadora urbana em Austin, Texas. Quando o município debate construir mais moradia, quem participa do debate são os moradores atuais. Os potenciais moradores, que viriam pra Austin se houvesse moradia, não aparecem na audiência pública. Mas eles contam. April tenta lembrar essas pessoas pros vereadores. Vale pra cidade, vale pra humanidade.

A frase final do capítulo é uma pergunta que vale você levar pro próximo episódio. Se juntos, sem coagir ninguém, sem sacrificar coisa importante, a gente conseguir construir uma sociedade onde mais vidas boas aconteçam, isso não seria melhor? Talvez muito melhor?

Aí termina a Parte três do livro, o argumento a favor das pessoas.

Próximo episódio começa a Parte quatro, o caminho à frente. Capítulo nove: Despovoamento não se resolve sozinho. Os autores enfrentam de cara o otimismo automático de quem acha que daqui a pouco a fertilidade volta. Te espero lá.""",
    },
    {
        "id": "10_nao_se_resolve_sozinho",
        "title": "Cap. 9 — Despovoamento não se resolve sozinho",
        "text": """Episódio dez. Despovoamento não se resolve sozinho.

Aqui começa a Parte quatro do livro, o caminho à frente. Os capítulos anteriores estabeleceram que estabilizar a população seria melhor do que despovoar. Os próximos perguntam como fazer isso, ou se faz mesmo. E o capítulo nove abre com a pergunta mais óbvia. Talvez o problema se resolva sozinho?

Spears e Geruso respondem direto. Não se resolve.

E eles te conduzem pelas três crenças populares de auto-correção, derrubam uma por uma, e explicam por que cada uma é falsa. Vou caminhar com você.

Primeira crença popular: a fertilidade vai parar em dois sozinha. Que dois é um número natural, mágico, ímã. Os autores já trataram disso lá atrás, mas reforçam. Em mil novecentos, a fertilidade global era acima de cinco. Em mil oitocentos, acima de seis. Na idade do bronze, ninguém sabe ao certo, mas certamente não era dois. Não existe força natural que ajuste a humanidade pra dois. Não existe termostato demográfico. Em todos os países que caíram abaixo de um vírgula nove, nunca voltaram. No Japão, fertilidade está em um vírgula quatro há trinta anos. Trinta anos. Não é convergência pra dois, é um platô abaixo de dois. Despovoamento crônico.

Segunda crença popular: os grupos com alta fertilidade vão herdar a Terra. Tipo Amish nos Estados Unidos, ultraortodoxos em Israel, alguma seita evangélica em algum lugar. Os autores publicaram um artigo na revista Demography especificamente sobre isso, porque ouvem essa pergunta muito. Vou resumir.

Os Amish têm hoje em média entre cinco e seis filhos por casal. Em mil novecentos e setenta e cinco, eram cinquenta mil. Hoje, trezentos e cinquenta mil. Crescimento absurdo. Se você projetar quatro por cento ao ano pra sempre, em duzentos anos são quase um bilhão de Amish. Resolvido?

Não. Dois motivos.

Primeiro, fertilidade alta também cai com o tempo. Em meados do século vinte, Amish tinham entre sete e nove filhos. Em dois mil e quinze, abaixo de seis. Continua caindo. Muçulmanos indianos historicamente tinham fertilidade mais alta que a média do país. Em mil novecentos e noventa e dois, quatro vírgula quatro. Em dois mil e onze, dois vírgula sete. Em dois mil e dezenove, dois vírgula quatro. Ou seja, o grupo de fertilidade alta da Índia tem fertilidade menor hoje do que a média indiana tinha há quinze anos. Alta fertilidade muda de definição. O que era oito é cinco. O que era cinco é três. E em algum momento é menos de dois.

Segundo motivo, e esse é mais bonito. Crianças não herdam perfeitamente a religião dos pais. Os autores trazem um exemplo histórico devastador, e pessoal. A vó católica do Mike Geruso, imigrante canadense francesa, veio pros Estados Unidos. Cresceu num enclave francês em Rhode Island, junto com onze irmãos. Falava francês, ia à missa duas vezes por semana, casa cheia de imagens. A mãe do Mike, Suzanne, também desse enclave, estudou em escola católica em francês nos Estados Unidos. Cresceu católica até a alma. Mas teve só dois filhos, usou anticoncepcional, esqueceu o francês. Tia Jeannie, irmã da Suzanne, teve zero filhos. A pirâmide se afunilou em duas gerações. E olha que era um grupo enorme, católicos franco-canadenses, milhões. Onde eles estão hoje? Diluídos.

Pra você acreditar que os Amish vão dominar o mundo, você precisa crer que ao longo de centenas e milhares de anos absolutamente nenhuma criança nascida em comunidade Amish vai escolher sair, e que a fertilidade alta vai parar de cair. Os dois pressupostos contradizem tudo que sabemos. Matematicamente possível, na prática implausível.

Terceira crença popular: a medicina vai resolver. Tratamentos de fertilidade, FIV, congelamento de óvulos. Os autores são pessoais aqui. Mike e a esposa April tentaram FIV. Dean e a esposa Diane tentaram FIV várias vezes. Eles conhecem o sofrimento. Defendem que tratamento de fertilidade seja acessível. Mas, pergunta separada: isso reverte fertilidade populacional?

Resposta: não. E a evidência mais clara vem de Israel. Em mil novecentos e noventa e quatro, o governo israelense tornou FIV cem por cento gratuita pra qualquer cidadã. Em dez anos, uso de FIV multiplicou por dez. Mas a fertilidade nacional, que era dois vírgula nove em mil novecentos e noventa e três, ficou em dois vírgula nove em dois mil e três. Três décadas depois, está em dois vírgula oito. Praticamente parada. Mulheres israelenses usaram FIV pra adiar maternidade, casar mais tarde, estudar mais, mas não pra ter mais filhos. Os economistas Naomi Gershoni e Corinne Low documentaram tudo num paper.

Spears e Geruso fazem uma analogia poderosa pra explicar isso. Imagina que existisse um botão do bebê. Você aperta, sai um bebê. Pode apertar quantas vezes quiser. Quantas vezes você apertaria?

Spears e Geruso conjecturam: poucas vezes. Mike e April apertariam mais do que tiveram, sim. Eles tentaram um segundo, tiveram aborto espontâneo, vão parar. Pra eles, o botão ajudaria. Mas pra maioria das mulheres no mundo, o limite não é biológico nem médico. O limite é a vida que elas escolhem ter. Mulher indiana média tem o primeiro filho com vinte e poucos anos, não está esbarrando em infertilidade. Mulher coreana média decide não casar.

Eles fazem outra analogia genial. E se inventassem uma tecnologia que eliminasse o fardo da amamentação? Que maravilha pra muitas mães. Pois é. Essa tecnologia já existe há décadas. Chama fórmula infantil. Em países ricos, com água potável, é tão boa quanto amamentar pra criança saudável. E não impediu queda da natalidade. Tecnologia ajuda, mas não cria desejo de ter filho.

E quanto a útero artificial, então? Em dois mil e dezessete, a Nature Communications publicou estudo onde um cordeiro foi gestado num saco plástico sobre uma chapa quente. Spears e Geruso dizem: ótimo. Será revolucionário pra alguns. Mas, mesmo que essa tecnologia chegue, alguém ainda precisa querer apertar o botão. Querer cuidar do bebê durante vinte anos depois. Tecnologia não cria vontade.

Aqui o capítulo faz uma comparação histórica preciosa. Em mil novecentos e trinta, John Maynard Keynes escreveu um ensaio prevendo que em cem anos, com o ganho de produtividade, as pessoas trabalhariam quinze horas por semana. Não trabalhamos. Keynes acertou que daria pra ter padrão de vida de mil novecentos e trinta com quinze horas hoje. Mas as pessoas não se contentaram com padrão de vida de mil novecentos e trinta. Suas ambições cresceram com suas possibilidades. Da mesma forma, tecnologia de fertilidade alivia a parte biológica, mas as ambições das mulheres cresceram com ela. Não temos pouca fertilidade porque não conseguimos ter filhos. Temos pouca fertilidade porque queremos outras coisas também.

A quarta e última crença popular que o capítulo enfrenta: bom, então quando despovoamento se tornar um problema sério, o governo vai agir. Esse é um capítulo inteiro adiante, sobre por que governos historicamente falharam. Mas Spears e Geruso adiantam um argumento aqui, e ele é elegante. Bebê é externalidade clássica.

Externalidade é quando uma pessoa toma uma decisão e outra pessoa paga a conta. Poluição é externalidade negativa. Descoberta científica é externalidade positiva. Quando você decide ter ou não ter um filho, os benefícios e custos sociais não recaem todos sobre você. Recaem sobre todo mundo. Sobre o futuro. Sobre o vizinho que vai depender do imposto que essa criança vai pagar daqui a quarenta anos. Sobre o desconhecido que vai usar a invenção que essa criança vai inventar daqui a sessenta. Você não captura nenhum desses benefícios. Você paga o custo todo. Por isso, do ponto de vista individual, é racional ter pouco ou nenhum filho.

E governos. Governos não conseguem agir porque bebês são free riders eleitorais. Eles não votam. Eles não pagam imposto agora. Eles vão pagar daqui a quarenta anos. Nenhum político vai gastar capital político hoje pra ter ganho daqui a quarenta anos, fora do seu mandato, fora da sua geração. A campanha o que queremos? Mudança! Quando? Daqui a trinta anos! Não funciona.

E pior. Se um país investir pesado em estabilizar a fertilidade, os benefícios em forma de ideias, vacinas, descobertas, vão pra humanidade inteira. Política unilateral não captura benefício nacional. É problema global.

A conclusão do capítulo é bonita, e termina com uma cena lindíssima. Reema, a irmã mais velha da enfermeira Seema, passou no concurso pra ser professora de enfermagem. Carreira garantida. As duas saem pra comemorar. Não conseguem ir tomar sorvete porque é caro, compram picolé laranja na rua. Caminham pelo bairro. Reema sabe que o concurso vai destravar o casamento dela, então fala sobre noivo, sobre planos, sobre filhos. Talvez dois, talvez não. Em Uttar Pradesh, irmãs casam por ordem de idade. Logo será a vez da Seema. O pai vai ligar com uma proposta. Se ela casar, sai do hospital. Sai do programa KMC. Não está nas mãos dela inteiramente.

Spears e Geruso terminam dizendo: durante toda a noite, Seema e Reema conversam sobre tudo isso, mas nenhuma das duas menciona externalidade populacional, sequestro de carbono, manejo de asteroide. Seria absurdo esperar que mencionassem. Elas têm muito o que conversar. As consequências populacionais coletivas existem, mas não cabe a essas duas mulheres carregarem essa preocupação na hora de escolher a própria vida. É tarefa coletiva.

Despovoamento é o pior tipo de externalidade. Global. Intergeracional. Política e cultural ao mesmo tempo. Nenhuma pessoa, família, empresa, país ou geração tem incentivo individual pra resolver. Igual mudança climática. Só se resolve por cooperação consciente. Não se resolve sozinho.

Próximo episódio, capítulo dez. Controle estatal não pode forçar estabilização. Vai ter história dura. Política do filho único na China. Política de aborto na Romênia comunista. Esterilizações forçadas. Vou te contar com cuidado. Te espero lá.""",
    },
    {
        "id": "11_controle_estatal",
        "title": "Cap. 10 — Controle estatal não pode forçar estabilização",
        "text": """Episódio onze. Controle estatal não pode forçar estabilização.

Esse é um capítulo curto, mas explosivo. O capítulo dez. Spears e Geruso atacam frontalmente uma das narrativas mais difundidas do mundo. A de que governos conseguem, através de coerção, controlar a natalidade. Eles vão te mostrar, com dados, que isso é uma lenda. E vão te mostrar por que essa lenda é perigosa, porque autoriza nacionalistas a sonhar com reverter despovoamento via restrição de aborto, contracepção, controle do corpo das mulheres.

Vou começar com o experimento mental que eles propõem. Eles te mostram um gráfico de fertilidade. Seis países, no mesmo período. China, Hong Kong, Romênia, Coreia do Sul, Taiwan e Tailândia. As curvas estão todas misturadas, sem rótulos. Spears e Geruso perguntam: qual desses países é a China?

Esse experimento veio de uma conversa com o economista Jesús Fernández-Villaverde, e antes dele de pesquisa do sociólogo Wang Feng. A política do filho único da China existiu entre mil novecentos e setenta e nove e dois mil e quinze. Trinta e seis anos de coerção. Abortos forçados. Esterilizações forçadas. Bebês ilegais sem direito a documento. Se essa política tivesse tido efeito gigante na fertilidade, a curva da China deveria saltar fora do grupo.

Mas não salta. A curva da China é indistinguível das curvas de Hong Kong, Coreia, Taiwan, Tailândia. Caem todas juntas, no mesmo ritmo, pelo mesmo formato. Sem rótulo, nenhum especialista consegue dizer qual é qual.

Por que isso? Porque o que estava caindo a fertilidade na China não era a política. Era a mesma força que caiu fertilidade em todo o Leste Asiático. Desenvolvimento econômico. Urbanização. Educação feminina. Alfabetização. Renda subindo. Saúde infantil melhorando. Tudo isso acontecia simultaneamente em todos esses países, com ou sem política do filho único.

A antropóloga americana Susan Greenhalgh, que estudou política populacional chinesa por décadas, viveu em vilarejos rurais nos anos oitenta e noventa. Logo na primeira página da obra magna dela, ela observa quase de passagem que a maior parte da queda chinesa parece ser por desenvolvimento socioeconômico, não pela política do filho único. Os números reforçam. Entre mil novecentos e oitenta e dois mil, a China caiu uma criança por mulher. A Índia, no mesmo período, caiu uma vírgula cinco criança por mulher, sem política do filho único. América Latina caiu uma vírgula seis criança por mulher. A China não foi a campeã da queda. Foi participante normal de um movimento global.

Não significa que a política do filho único não fez mal. Fez muito mal. Spears e Geruso são absolutamente claros. Houve aborto forçado, esterilização forçada, mulher escondida do Estado pra ter filho. Houve milhões de crianças nascidas ilegalmente, sem direito a documento, sem direito a escola pública. Há, até hoje, um desequilíbrio de gênero pesado, com sete homens jovens pra cada seis mulheres jovens, distorcendo casamento e família.

Mas, e esse é o ponto, o crime não foi efetivo no que prometia. Não foi a política que reduziu a fertilidade chinesa a um vírgula dois. Foi a vida moderna fazendo o que a vida moderna faz em todo lugar. A China chegou ao mesmo destino dos vizinhos. Talvez por uma trilha mais brutal, mas ao mesmo destino.

A propaganda oficial do Partido Comunista diz que a política preveniu quatrocentos milhões de nascimentos. Spears e Geruso dizem: não acredite. Acredite nos dados.

Agora, do lado oposto. E se um governo tentar forçar pra cima, em vez de pra baixo? Aí entra a Romênia.

Em mil novecentos e sessenta e seis, a fertilidade romena era um vírgula nove. Abaixo do nível de reposição. Em mil novecentos e sessenta e sete, o ditador Nicolae Ceaușescu emitiu o Decreto setecentos e setenta. Aborto, que era amplamente legalizado havia uma década, foi criminalizado. A fertilidade pulou pra três vírgula seis em um ano. Quase dobrou.

Pareceu funcionar. Por três anos. Mas em mil novecentos e setenta e um, já tinha caído pra meio caminho da posição original. Em mil novecentos e oitenta, três quartos do caminho de volta. E o governo não relaxou. Pelo contrário. Nos anos oitenta, redobrou. Exames pélvicos obrigatórios em algumas mulheres no trabalho. Tortura institucional. Mulheres romenas morrendo em abortos clandestinos numa taxa muito maior que em qualquer país comparável.

A fertilidade ficou um pouco acima do nível inicial de mil novecentos e sessenta e seis. Pouco. Até dezembro de mil novecentos e oitenta e nove, quando o Ceaușescu foi deposto e fuzilado contra um muro. A política caiu junto.

Spears e Geruso fazem uma observação seca aqui. Se um regime é tão horrível que termina executado contra um muro por levante popular, ninguém deveria fingir que esse regime descobriu uma fórmula sustentável pra elevar fertilidade. Efeito breve, custo humano gigante, e não muda o pico.

E olha o detalhe revelador. No gráfico que eles mostraram no começo do capítulo, a Romênia também está lá, junto com os países asiáticos. Não dá pra ver qual é a Romênia também. Política antinatal extrema da China, política natalista extrema da Romênia, e nenhuma das duas curvas é distinguível das vizinhas.

A conclusão é forte. Controle populacional, de qualquer direção, simplesmente não controla a população no longo prazo. Influencia no curto prazo, sim. Causa sofrimento, sim. Mas o trem da fertilidade, depois do desenvolvimento, segue sua rota independente das tentativas de manipulação.

Aí o capítulo me oferece o caso pessoal do livro. Lembra de April, a esposa do Mike, capítulo quatro? Ela queria o segundo filho. Engravidou. A Suprema Corte derrubou Roe versus Wade no mesmo mês. Texas restringiu aborto. April teve aborto espontâneo, com medo de não conseguir atendimento adequado. Decidiu não tentar mais. Restringir aborto, no caso dela, reduziu o número de filhos que ela gerou. Não aumentou.

Spears e Geruso também citam a escritora Ursula Le Guin, que disse que ela só teve três filhos porque, primeiro, fez um aborto. Se ela tivesse sido forçada a ter aquela primeira gravidez não planejada aos vinte anos, teria largado a faculdade, não teria casado com o marido que ela amava, não teria viajado pra França, não teria escrito os livros. E seus três filhos planejados não teriam nascido. Cada história individual.

E olha os dados. A Coreia do Sul, único país rico com aborto fortemente restrito até dois mil e dezessete, segundo o Instituto Guttmacher, tem hoje a menor fertilidade do mundo desenvolvido. Restrição de aborto na Coreia não elevou nada. E em dois mil e dezenove, a Corte Constitucional sul-coreana declarou a restrição inconstitucional. Caiu.

Tem outro padrão histórico, e esse é o mais importante. Pra onde o mundo está caminhando? Para mais liberdade reprodutiva, não menos. A Irlanda legalizou aborto em referendo dois mil e dezoito com sessenta e seis por cento dos votos. O Decreto setecentos e setenta da Romênia caiu junto com Ceaușescu. Desde mil novecentos e noventa e quatro, só quatro países restringiram aborto: El Salvador, Nicarágua, Polônia e Estados Unidos.

E mesmo onde restringem, mulher contorna. A pesquisadora Abigail Aiken, colega dos autores na Universidade do Texas, documentou que aborto via medicamento autoadministrado pelo correio cresceu muito desde a queda do Roe. Telemedicina entrega remédios pra qualquer lugar. A revolução da liberdade reprodutiva é como genie fora da garrafa. Não volta.

Spears e Geruso terminam com uma mensagem que eu acho moralmente forte. Por que insistir em desmontar essa lenda? Porque a lenda autoriza fantasia. Se você acredita que controle populacional funciona, você passa a aceitar que talvez restringir aborto possa salvar o país do despovoamento. E essa fantasia é exatamente o que nacionalistas, fundamentalistas e autoritários estão sussurrando agora. Spears e Geruso querem cortar essa raiz. Coerção não funciona. Coerção é imoral e ineficaz. As duas coisas precisam ser ditas juntas, porque uma sem a outra é insuficiente.

A conclusão lógica é simples. Se controle não funciona, e se despovoamento é problema sério, então só sobra um caminho. O caminho do incentivo, da liberdade, do apoio à parentalidade. É o que o próximo capítulo enfrenta, e é a questão mais prática do livro inteiro: dinheiro funciona? Bônus por bebê funciona? Subsídio funciona? Te conto no próximo episódio.

Próximo episódio, capítulo onze. Dinheiro é a resposta? Te espero lá.""",
    },
    {
        "id": "12_dinheiro_e_resposta",
        "title": "Cap. 11 — Dinheiro é a resposta?",
        "text": """Episódio doze. Dinheiro é a resposta?

Esse é, na minha opinião, o capítulo mais útil pra qualquer pessoa que quer entender debate de família e fertilidade hoje. Capítulo onze. A pergunta é direta. Se controle não funciona, e despovoamento é problema sério, basta o governo dar dinheiro pra quem tem filho? Bolsa criança? Licença remunerada grande? Creche gratuita? Spears e Geruso vão te mostrar que dinheiro alivia, mas não estabiliza. E vão te dar uma teoria alternativa, a teoria do custo de oportunidade, que é elegante.

Vamos por etapas. A primeira pergunta. Será que pessoas têm menos filhos hoje porque filho ficou caro demais? A chamada hipótese da inacessibilidade. Os autores fazem três testes empíricos. Vou chamar de strike um, strike dois e strike três, como eles fazem.

Strike um. Comparar países. Se filho é caro demais, países ricos deveriam ter menos filhos por falta de bolso. Mas a verdade é o oposto. Países ricos têm menos filhos, países pobres têm mais. Texas tem renda muitas vezes maior que Uttar Pradesh, e fertilidade do Texas é um vírgula oito, de Uttar Pradesh é dois vírgula três. Latim América e Caribe, renda média, fertilidade um vírgula oito. Subsaariana, mais pobre região do mundo, fertilidade acima de quatro. Se fosse o bolso, seria o contrário.

Strike dois. Comparar o mesmo país ao longo do tempo. Conforme o país enriquece, o que acontece com a fertilidade? Cai. Sempre. Em todos os continentes. Em todas as regiões. A flecha aponta pra baixo e pra direita. Mais renda, menos filhos. A hipótese da inacessibilidade prevê o oposto.

Strike três. Comparar estados americanos onde os preços de creche e moradia subiram em ritmos diferentes. Os economistas Melissa Kearney, Phil Levine e Luke Pardue fizeram esse estudo. Resultado: Massachusetts viu o preço de creche subir mais de quatro mil e quinhentos dólares por ano por criança. Delaware viu menos de mil. Os dois estados tiveram a mesma queda de fertilidade. O distrito de Columbia teve aumento de aluguel sete vezes maior do que Connecticut ou Oklahoma. Mesma queda de fertilidade nos três. Sem padrão.

Hipótese da inacessibilidade. Eliminada. Strike um, strike dois, strike três. Está fora.

Mas Spears e Geruso são honestos. Eles dizem: existe alguma coisa real no que as pessoas estão sentindo quando dizem que filho está caro. O que é? Resposta deles, e é a chave do capítulo: não é custo monetário. É custo de oportunidade.

Custo de oportunidade é o que você abre mão pra ter outra coisa. Pode ser dinheiro, mas pode ser tempo, energia, ambição profissional, vida social, viagens, projetos pessoais. A economista Emily Oster, de Brown, escreveu o livro Cribsheet sobre parentalidade. Ela diz: tempo gasto com filho é tempo não gasto com outra coisa. Não é só sobre como fazer melhor, é sobre o que você está abrindo mão.

E aí vem a sacada do capítulo. Quanto melhor o mundo fica, mais alto o custo de oportunidade de ter filho. Não porque filho ficou pior, mas porque tudo mais ficou melhor.

Vou repetir. Não é que filho ficou pior. É que tudo mais ficou melhor.

Carreira ficou melhor. Em mil novecentos e sessenta, um terço das jovens americanas esperava trabalhar a vida toda. Em mil novecentos e oitenta, oitenta por cento. Hoje, praticamente todas. Carreira virou parte da identidade. Abrir mão da carreira pra criar filho passou a custar muito.

Lazer ficou melhor. Maratona com vinte ligações de Netflix. Viagem internacional barata. Restaurante para todos os gostos. Ler qualquer livro do mundo no Kindle.

Saúde ficou melhor. Tênis novo permite que professor de quarenta anos corra meia maratona. Articulação artificial permite cirurgia que dá mais dez anos de mobilidade.

Tudo ficou melhor. Tudo. E filho compete com tudo isso.

E aí entra um número de exemplo. Spears e Geruso citam um estudo. Sueca, por exemplo. Suécia tem creche subsidiada quase de graça. Licença parental dezesseis meses paga, dividida entre o casal. Plano de saúde universal. Gasta com benefício familiar duas vezes a fração do PIB que os Estados Unidos gastam. Resultado: fertilidade sueca de um vírgula sete seis. Fertilidade americana, com muito menos benefício, de um vírgula sete três. Praticamente igual. A política familiar generosa da Suécia não levou fertilidade pra dois.

E não é só Suécia. Dinamarca, Noruega, Finlândia, todos com benefícios robustos, todos com fertilidade abaixo de dois. Em alguns casos, abaixo da americana.

A pergunta fica óbvia. Por que dinheiro não funciona?

A resposta é que dinheiro é caro demais pra cobrir o custo de oportunidade real. Pra dar conta, o cheque teria que ser não três mil dólares, não dez mil dólares, mas alguma coisa que compense largar carreira inteira, anos de juventude, projetos pessoais, viagens. Pra muita gente, isso é impagável por qualquer governo.

E mais. Spears e Geruso observam um efeito interessante. Quando o governo dá um bônus por bebê, casais tendem a antecipar o nascimento. Ter o bebê aos trinta e um em vez de aos trinta e quatro. Mas o total ao longo da vida da mulher não muda. Ou seja, o cheque acelera, não aumenta. E pra evitar despovoamento, o que conta é o total ao longo da vida.

Tem outro efeito perverso. Em sociedades mais ricas, a régua do que é parentalidade aceitável sobe. Em mil novecentos e sessenta, a casa americana média tinha quatro vírgula quatro cômodos. Em dois mil e vinte, cinco vírgula seis. Filho dividindo cama com irmão era padrão pro pai do Mike. Hoje é considerado pobreza extrema. Cada cheque entregue precisa cobrir um padrão de parentalidade sempre mais alto. É uma esteira que sobe.

Aí o livro entra num exemplo crucial pra você não cair em explicações fáceis. A Índia. Onde fertilidade está abaixo de dois e mulher quase não trabalha fora de casa. Setenta e cinco por cento das mulheres indianas não tem emprego remunerado. Só uma em oito tem emprego classificado como profissional ou técnico. A média de idade no primeiro filho é início dos vinte. Quase todo mundo casa. Quase ninguém divorcia. A religião é praticada cotidianamente.

Por essa lógica simplista, a Índia deveria ter fertilidade altíssima. Mas tem dois. Por quê? Spears e Geruso dizem: porque o custo de oportunidade ali não é só carreira. É sogra. É escola dos filhos. É moradia. É deslocamento. É expectativa social do que é vida digna. Mesmo onde a mulher não tem carreira, vida moderna gera custo de oportunidade. A Índia mostra que o fenômeno é mais profundo que feminismo, mais profundo que neoliberalismo, mais profundo que qualquer narrativa simples.

E isso leva Spears e Geruso a uma observação dura. Eles dizem o seguinte: ninguém tem a grande teoria unificada da queda da fertilidade. Ninguém. Tem-se uma constatação. Conforme a vida melhora pras pessoas, em muitas dimensões, ter filho compete com mais coisas boas. Mas a fórmula precisa de cada lugar, de cada cultura, de cada geração, ninguém sabe.

Eles listam várias hipóteses que circulam e mostram que cada uma quebra em algum lugar.

Capitalismo? Mas Coreia do Norte e Cuba têm fertilidade baixa. Países nórdicos socialistas têm fertilidade baixa.

Casamento em queda? Mas Índia, Nepal e China casam muito e têm fertilidade baixa.

Feminismo? Mas Coreia do Sul tem o maior pay gap da OCDE e a menor fertilidade.

Individualismo ocidental? Mas Ásia oriental e sudeste asiático tem fertilidade ainda menor que Europa, América do Norte, Austrália.

Contracepção? Mas a fertilidade caía na França antes mesmo da Independência dos Estados Unidos.

Religião em queda? Mas a Índia religiosíssima tem fertilidade abaixo de dois.

Spears e Geruso resumem com humildade. Ninguém ainda entende totalmente por que a fertilidade caiu em todo lugar abaixo de dois. Mudança climática a gente entende. Sabe que CO dois aquece. Sabe por que CO dois é emitido. A questão é só vontade política. Despovoamento, a gente sabe o que acontece com fertilidade baixa, mas não sabe por que ela é baixa.

E essa diferença, eles dizem, é o que torna o problema do despovoamento ainda mais difícil de resolver que o problema climático.

Mas, e essa é a parte que eu acho importante levar pro próximo episódio, mesmo sem teoria geral, eles dizem que sabem alguma coisa. Custo de oportunidade vai continuar subindo. Vida vai continuar melhorando. Outras opções vão competir cada vez mais com filho. Cheque do governo, no tamanho que alguém já tentou, não cobre.

Então, se controle não funciona e dinheiro não funciona, o que sobra? Sobra uma coisa. Sobra repensar a sociedade. Sobra valorizar parentalidade culturalmente. Sobra redistribuir o trabalho de cuidar. Sobra criar um mundo onde escolher ter filho não signifique abrir mão de tudo. Sobra ambição grande.

Esse é o título do próximo capítulo. Aspire Bigger. Almeje mais alto. O capítulo de fechamento do livro, com a tese mais propositiva e talvez a mais utópica. Te espero lá.""",
    },
    {
        "id": "13_almeje_mais_alto",
        "title": "Cap. 12 — Almeje mais alto",
        "text": """Episódio treze. Almeje mais alto.

Esse é o último capítulo do corpo principal do livro. Capítulo doze. Aspire Bigger, almeje mais alto. E os autores são absolutamente honestos sobre o que esse capítulo é e o que ele não é.

Logo na primeira linha. Esse capítulo não é a Solução com S maiúsculo. Não tem solução. Não ainda. Não se solução significa um plano à prova de falha que ambicioso, detalhado, e capaz de tirar a humanidade do despovoamento. Ninguém tem. O desafio é novo demais.

O que tem? Uma direção. Uma visão. E uma postura, que é a tese moral do livro inteiro. A postura de que vale a pena tentar.

Os autores voltam à enfermeira Seema pela última vez. Eles contam que parte do trabalho dela, o pior assignment, é fazer ligações pra famílias que saíram do hospital. As ligações são duras. Muito pai não quer falar com enfermeira. Famílias que abandonaram o programa Mãe Canguru contra recomendação médica. Bebês fragilíssimos em casa. Seema sentada num quartinho dos fundos do hospital, computador na frente, ligando uma família por vez. Tentando trazer o bebê de volta. Os supervisores dela dizem: eu não sei como ela faz, mas se mandar a Seema, o bebê volta.

Spears e Geruso usam isso como metáfora. Não é o plano perfeito. É a teimosia humana. É a vontade de não desistir. Em mil novecentos e setenta e oito, dois pediatras em Bogotá inventaram o cuidado canguru sem ter um plano de quarenta e sete anos pra implementar globalmente. Não imaginavam computadores, planilhas, telefones com geolocalização. Mas tiveram compromisso. Daí saiu tudo o resto.

A tese do capítulo é essa. Vai dar pra resolver despovoamento, mas exige que a gente comece com visão e compromisso, antes do plano. Como começam transformações grandes.

Pra te convencer que mudança radical é possível, eles trazem três exemplos históricos que provam isso. Os três mudaram drasticamente o que as pessoas esperam da vida.

Um. Escola pública gratuita, obrigatória e universal. Em mil setecentos e sessenta e três, Frederico o Grande emitiu o Regulamento Escolar Geral, na Prússia. Estabeleceu oito anos de educação básica, governo paga, todos os meninos e meninas obrigados a frequentar. Em mil setecentos e sessenta e três, quase ninguém no mundo achava razoável mandar criança pra escola por oito anos. Escola era pra elite. Hoje, em qualquer país desenvolvido, ninguém imagina criança sem oito anos de escola. O que era esquisitice virou direito básico em poucas gerações.

Dois. Esgoto. Em mil oitocentos e cinquenta, Massachusetts tinha educação universal. Mas Boston ainda jogava fezes em fossas urbanas. Londres construiu esgoto no final do século dezenove. Paris só conectou a maioria das casas ao esgoto no século vinte. Antes disso, mesmo gente rica em cidade rica usava penico, jogava na rua. Antes do esgoto, ninguém esperava banheiro descarga como direito básico. Hoje, ninguém aceita morar sem banheiro descarga. O conceito de vida digna foi transformado.

Três. Previdência social e plano de saúde pra idoso. Nos Estados Unidos, o primeiro cheque de aposentadoria foi pago em mil novecentos e quarenta. Vinte e dois dólares e cinquenta e quatro centavos. Antes disso, muitos idosos americanos morriam na pobreza. Hoje, todo americano espera que ao envelhecer, tenha aposentadoria, plano de saúde, remédio caro pago. Foi mudança radical em três gerações.

O argumento dos autores é o seguinte. Cada uma dessas transformações exigiu investimento gigantesco, mudança cultural, debate político, sacrifício de privilégios estabelecidos. Cada uma mudou o que as pessoas esperam de uma vida boa. Cada uma virou óbvia depois. E todas começaram com gente que decidiu que ia ser melhor assim.

Por que não pode ser assim com parentalidade?

Imagine uma sociedade onde cuidar de criança em férias escolares e dia sem aula simplesmente não é estresse da mãe nem do pai. Tem solução pública. Como creche. Como escola. Como hospital. Onde licença parental não é vinte semanas heroicas e depois tudo de volta no colo da mãe. Onde homem cuida de filho na média igual a mulher, não como exceção elogiável. Onde quem não tem filho contribui via imposto, via tempo voluntário, via vizinhança ativa, com a criação dos filhos dos outros, porque entendeu que isso é interesse coletivo.

Spears e Geruso dizem: isso não vem de altruísmo individual. Vem de política pública. De cultura redesenhada. De projetos coletivos. Eles dão um exemplo simples. Eles mesmos pegam ônibus pro trabalho. Mike vai pra Casa Branca pegando ônibus. Dean vai pro centro de pesquisa pegando ônibus. Não pegam por virtude. Pegam porque pegar ônibus é a opção mais conveniente, mais barata, mais confortável pra eles. E pegar ônibus, secundariamente, é bom pro clima. A virada do clima passa quando a opção que é melhor pra você é também a opção que é melhor pra o planeta. Por design. Não por altruísmo.

Pra parentalidade, a tese é a mesma. Não adianta esperar que casais sacrifiquem mais por dever cívico. O sistema precisa ser redesenhado pra que escolher ter filho seja uma boa decisão pessoal pra quem escolhe. Pra que parental não seja punição de carreira, perda de viagens, encolhimento de identidade. Hoje, escolher ter filho é abrir mão de muita coisa boa. Os autores dizem: pra reverter o despovoamento, a meta é construir uma sociedade onde escolher ter filho continue sendo escolha, mas onde a escolha seja atrativa.

E aqui aparece a frase que eu mais gostei do capítulo. O que vai moldar o longo prazo é o que as pessoas do futuro vão aspirar. E a gente tem motivo pra otimismo. Por quê? Porque pessoas do futuro ainda não aspiram nada. Elas ainda não existem. Aspirações são formadas pelo mundo em que se nasce. A geração que nascer numa sociedade onde parentalidade é apoiada, atraente, redistribuída, vai aspirar mais filhos. Não por ímã genético, mas por mimese cultural.

Spears e Geruso fazem ainda uma observação importante sobre custo. Sim, transformar parentalidade vai custar dinheiro. Vai custar redesenho econômico. Vai mexer em privilégios estabelecidos. Mas, eles perguntam, comparado com a alternativa, ou seja, despovoamento e tudo que ele implica, esse custo é grande?

E eles dão dado que reforça. Décadas de pesquisa em economia mostram que investimento em criança gera retorno gigante. Criança saudável, bem-educada, vira adulto mais produtivo, paga mais imposto, depende menos de assistência social. Sem contar inovação e progresso. Estabilizar população talvez se pague no longo prazo.

E aí, no fim do capítulo, eles fazem uma simulação numérica linda. Tem o gráfico do pico, lembra. E aqui eles plotam três cenários possíveis. Em todos, a fertilidade global eventualmente volta pra dois. A única diferença é quando isso começa. Em dois mil e cento e vinte e cinco. Em dois mil e cento e cinquenta. Em dois mil e cento e setenta e cinco. Cinquenta anos a mais ou a menos de espera.

Resultado. Se a virada começar em dois mil e cento e vinte e cinco, a humanidade estabiliza em nove vírgula três bilhões. Se começar em dois mil e cento e setenta e cinco, cinquenta anos depois, estabiliza em seis vírgula oito bilhões. Diferença de dois vírgula cinco bilhões de pessoas, pra sempre, ano após ano. Pra cada década de atraso, oito por cento a menos no tamanho estabilizado.

Dois vírgula cinco bilhões de pessoas. Pra você ter ideia, isso é mais ou menos a humanidade inteira em mil novecentos e cinquenta. É menos do que tinha quando o pai do Dean e o pai do Mike nasceram. É como se cinquenta anos de inação fizessem a humanidade perder, pra sempre, todo o tamanho que ela tinha em mil novecentos e cinquenta.

Os autores fecham com uma analogia que já apareceu antes. Os cientistas climáticos dos anos cinquenta. Em mil novecentos e cinquenta e oito, montaram um observatório em Mauna Loa, no Havaí, pra medir CO dois na atmosfera. Em mil novecentos e cinquenta e oito, ninguém ia fazer nada sobre o aquecimento. Ia demorar décadas até começar a haver vontade política. Mas eles começaram, plotaram o gráfico, juntaram dados, formaram um corpo de conhecimento. Sem aquele observatório de Mauna Loa, hoje a humanidade estaria na escuridão. Por causa dele, a discussão sobre clima começou décadas antes do necessário.

Despovoamento, dizem Spears e Geruso, é igual. A virada talvez aconteça daqui a sessenta anos. Mas se quem começa o trabalho hoje for o equivalente aos cientistas de Mauna Loa, daqui a sessenta anos a humanidade vai ter ferramenta, terá tido debate, terá tido ciência. Sem isso, vai chegar lá às cegas.

A última frase do livro, fora o apêndice, é solene. Esperar pra depois do pico pra aprender como responder ao despovoamento seria tão imprudente quanto esperar queimar a última onça de carvão pra começar a responder à mudança climática. Mudança climática nos ensinou que importância pode vir com incerteza. E que incerteza não te tira do dever de esperar, aprender e organizar. A humanidade está num caminho de despovoamento. Isso nos chama, de novo, pra esperar, aprender, organizar.

E aí termina o argumento principal do livro.

No próximo episódio, o último da série, vamos pro Apêndice Repugnante. Esse é um capítulo bônus pra quem quer mergulhar na filosofia. Spears e Geruso vão encarar de frente a famosa conclusão repugnante do filósofo Derek Parfit. Aquela ideia: se mais vidas boas é melhor, então um mundo de trilhões de pessoas vivendo só um pouquinho acima do limite suportável é melhor que um mundo de oito bilhões vivendo ótimo? Vou te contar com cuidado. Te espero lá pra fechar a série.""",
    },
    {
        "id": "14_apendice_repugnante",
        "title": "Apêndice — A conclusão repugnante",
        "text": """Episódio quatorze. Apêndice repugnante.

Esse é o último episódio da série. Pode soar estranho terminar com um apêndice. Mas Spears e Geruso fazem esse apêndice porque sabem que qualquer leitor curioso de ética populacional vai bater na questão que ele responde. É a famosa conclusão repugnante do filósofo Derek Parfit, formulada nos anos oitenta. E é uma das objeções mais comuns à ideia de que mais vidas boas é melhor. Vou explicar com calma.

Primeiro, o problema. Imagine duas humanidades possíveis no futuro.

Humanidade A. Um bilhão de pessoas. Cada uma com qualidade de vida nota dez na escala cósmica. Vidas excepcionalmente boas. Cultura, arte, amor, realização, saúde, prosperidade.

Humanidade B. Um trilhão de pessoas. Mil vezes mais gente. Cada uma com qualidade de vida nota dois. Vidas só um pouquinho positivas. Comer batata, ouvir música genérica, viver sem grandes sofrimentos, sem grandes alegrias. Apenas suficientemente boa.

Qual é melhor?

Se você somar o bem-estar total, a humanidade B vence. Um trilhão vezes dois é maior que um bilhão vezes dez. Pelo critério do bem-estar total, mais gente vivendo só um pouquinho bem é preferível a menos gente vivendo otimamente.

Parfit, filósofo brilhante, no livro Reasons and Persons de mil novecentos e oitenta e quatro, achou essa conclusão repugnante. Em sentido literal. Algo no nosso senso moral grita. Como assim, um trilhão de pessoas medíocres é melhor que um bilhão de pessoas com vidas dignas? Parfit chamou isso de Conclusão Repugnante e usou como argumento contra o critério do bem-estar total.

Por quatro décadas, filósofos perderam o sono com isso. The Economist, Nate Silver, Peter Singer, Matt Yglesias, todos escreveram sobre. Virou meme intelectual.

Spears e Geruso fizeram o apêndice por dois motivos. Primeiro, porque qualquer leitor minimamente curioso vai esbarrar nessa ideia, e seria desonesto ignorá-la. Segundo, porque eles têm uma posição clara, e querem desarmar essa ideia. Eles dizem o seguinte. A Conclusão Repugnante não é um problema único do critério do bem-estar total. Não é argumento que distingue uma teoria moral da outra. E, mais importante, não tem nenhuma relevância pra discussão prática entre despovoamento e estabilização. Vou destrinchar.

O argumento original do Parfit pressupunha que comparamos só os dois grupos hipotéticos. Humanidade A com um bilhão, humanidade B com um trilhão. Nada mais. Aí, no critério da média, A vence, porque a média é dez. No critério do total, B vence. Parfit usou isso pra dizer: o critério da média é certo, o critério do total é errado.

Spears e Geruso, junto com o filósofo Mark Budolfson, mostraram numa publicação acadêmica que isso é uma armadilha do exemplo. Eles dizem: por que comparamos só os dois grupos isolados? Não existe outro grupo? E aí, eles incluem um detalhe historicamente verdadeiro. Cento e vinte bilhões de pessoas já viveram nessa Terra ao longo da história humana. Setenta por cento dessas vidas tiveram qualidade nem tão boa. Comida ruim, doença, morte precoce. Vamos atribuir a essas cento e vinte bilhões de vidas passadas uma qualidade média de um.

Agora recalcule a média no cenário humanidade A versus humanidade B, incluindo o grupo de fundo das cento e vinte bilhões de pessoas que existiram.

Cenário A. Cento e vinte bilhões de pessoas com nota um, mais um bilhão com nota dez. A média sobe pouquinho de um, porque um bilhão de notas dez não diluem suficientemente cento e vinte bilhões de notas um.

Cenário B. Cento e vinte bilhões com nota um, mais um trilhão com nota dois. A média sobe muito mais, porque um trilhão de notas dois move o ponteiro com força.

Resultado. No critério da média, com o grupo de fundo incluído, a humanidade B também vence. Não é só o bem-estar total que aponta pra B. É o bem-estar médio também. As duas teorias coincidem.

Ou seja. A Conclusão Repugnante não é privilégio do bem-estar total. Ela emerge de praticamente qualquer teoria que faz comparações sensatas, quando o cenário inclui grupo de fundo realista. O incômodo emocional com a conclusão pode ser sincero. Mas ele não justifica abandonar bem-estar total em favor de bem-estar médio. Os dois ônibus levam pra mesma estação.

Spears e Geruso citam um economista chamado Yew-Kwang Ng, que provou matematicamente, no final dos anos oitenta, que qualquer teoria moral completa que respeite duas regras óbvias, que igualdade não é ruim e que adicionar vida boa não piora o mundo, vai gerar alguma versão da Conclusão Repugnante. O filósofo Gustaf Arrhenius depois generalizou. Ou seja, ou você abandona princípios morais bem básicos, ou você convive com a Conclusão Repugnante. Os autores defendem que conviva.

Mas eles vão além. Defendem que a Conclusão Repugnante talvez nem seja tão repugnante quanto Parfit fez parecer.

Por quê? Algumas razões.

Primeira. A vida nota dois, vida só um pouquinho positiva, é melhor que não existir. Por definição. Spears e Geruso dizem: você está tão acostumado com sua vida nota cinco que parece insulto chamar nota dois de boa. Mas pra quem vive a vida nota dois, ela é melhor que não existir. Pra quem não vive, não existe diferença a sentir, porque não existe sujeito. Adicionar muitas vidas nota dois é adicionar bem ao universo. Repugnante é palavra forte demais.

Segunda. Filósofos como Johan Gustafsson argumentaram que nossas intuições falham diante de números gigantescos. A gente não consegue sentir um trilhão de pessoas no estômago. A gente sente um bilhão e o trilhão é igual. A nossa intuição é treinada pra escala humana, não pra escala cósmica. Confiar nela como ferramenta moral quando os números atingem ordens de grandeza extremas é arriscado. Precisamos de lógica, não só de sentimento.

Terceira. Economistas, em geral, ficam menos incomodados com a Conclusão Repugnante do que filósofos. Por quê? Porque economistas estão acostumados com trade-offs, valor esperado, marginalidade. Pra um economista, ouvir que um trilhão de notas dois supera um bilhão de notas dez é apenas aritmética básica de soma. Não é monstruoso.

Quarta, e essa é a mais elegante. A Conclusão Repugnante é sobre trade-offs entre qualidade e quantidade. Cenário onde uma sobe e a outra cai. Mas o ponto central do livro inteiro é que entre despovoamento e estabilização não há trade-off. As duas dimensões andam juntas. Estabilização gera mais inovação, mais especialização, mais economia de escala, mais combate ao asteroide, mais sequestro de carbono, mais ciência, mais vacina. Estabilização é ganho em qualidade. E é ganho em quantidade. É win-win.

A Conclusão Repugnante só ataca alguém que defende quantidade contra qualidade. Spears e Geruso não defendem isso. Eles defendem que num mundo estabilizado, tanto a qualidade quanto a quantidade são maiores. Não tem trade-off pra perder o sono.

Spears e Geruso terminam o apêndice com uma serenidade quase elegante. A Conclusão Repugnante, dizem eles, é um quebra-cabeça filosófico interessante. Mas que tem zero implicação prática pra escolha que a humanidade enfrenta entre despovoar até virar pó ou estabilizar em alguma escala próspera.

E aqui se fecha a série. Catorze episódios. Vamos passar rapidinho pelo arco.

Episódio um, prólogo. A humanidade está num caminho de despovoamento. Não estabilização. Despovoamento exponencial.

Episódio dois, capítulo um. O pico. Dois mil e doze foi possivelmente o ano de mais nascimentos da história humana. Estamos no topo da agulha.

Episódio três, capítulo dois. Linha divisória. Dois é o número mágico. Abaixo de dois, decaimento exponencial. Mortalidade infantil esgotou. Só natalidade conta agora.

Episódio quatro, capítulo três. Pessoas e planeta. Mais gente não é mais poluição. Despovoamento não resolve clima. Net zero resolve clima.

Episódio cinco, capítulo quatro. Corpo das outras. Estabilização é compatível com igualdade de gênero. Coerção não funciona e nem é necessária.

Episódio seis, capítulo cinco. Mundo imperfeito. Vida sempre foi imperfeita. Vida hoje é melhor que de Benjamin Franklin. Trazer filho a um mundo cheio de problemas é decisão racional.

Episódio sete, capítulo seis. Progresso vem de pessoas. Ideias não se gastam. Inovação é não rival. Menos pessoas é menos ideias.

Episódio oito, capítulo sete. Asteroides e custos fixos. Mais gente paga custos fixos mais facilmente. Asteroide, vacina, sequestro de carbono, todos custos fixos.

Episódio nove, capítulo oito. Mais bom é melhor. Adicionar vida boa é adicionar bem ao mundo. Liberdade reprodutiva e estabilização convivem.

Episódio dez, capítulo nove. Não se resolve sozinho. Amish não dominarão o mundo. FIV não reverte. Tecnologia não reverte. Bebê é externalidade global.

Episódio onze, capítulo dez. Controle estatal não funciona. Política do filho único da China não foi a causa. Decreto setecentos e setenta da Romênia não funcionou. Restringir aborto não eleva fertilidade.

Episódio doze, capítulo onze. Dinheiro não é a resposta. Suécia não tem fertilidade maior. Custo de oportunidade explica. Tudo melhorou exceto a parentalidade.

Episódio treze, capítulo doze. Almeje mais alto. Escola universal, esgoto, previdência, tudo já foi impossível antes de virar óbvio. Parentalidade pode ser o próximo. Mas precisa de visão e compromisso.

E hoje, episódio quatorze, apêndice. Conclusão repugnante é uma elegância filosófica, mas não derruba a tese de que estabilizar é melhor que despovoar.

A tese final. A humanidade está num caminho que termina em desaparecimento. A gente tem algumas décadas. Não é crise. É chamado de longo prazo, igual ao clima foi nos anos cinquenta. E quem começa a conversa hoje, importa.

Você ouviu uma série inteira sobre o livro After the Spike de Dean Spears e Michael Geruso. Se algum tema te tocou, leia o livro. E pense. Obrigado por ouvir até aqui.""",
    },
]

print(f'✓ {len(PODCASTS)} podcasts carregados')
for p in PODCASTS:
    print(f'  - {p["id"]}: {p["title"]} ({len(p["text"])} chars)')


✓ 14 podcasts carregados
  - 01_prologo: Prólogo — o livro que pede pra você pensar de novo (6923 chars)
  - 02_o_pico: Cap. 1 — O pico (6693 chars)
  - 03_linha_divisoria: Cap. 2 — A linha divisória entre crescimento e decadência (8211 chars)
  - 04_pessoas_e_planeta: Cap. 3 — O que as pessoas fazem ao planeta (8744 chars)
  - 05_corpos_dos_outros: Cap. 4 — A população começa no corpo dos outros (9241 chars)
  - 06_mundo_imperfeito: Cap. 5 — Adicionar novas vidas a um mundo imperfeito (8767 chars)
  - 07_progresso_vem_de_pessoas: Cap. 6 — Progresso vem de pessoas (8854 chars)
  - 08_desviando_asteroide: Cap. 7 — Desviando do asteroide (9651 chars)
  - 09_mais_bom_e_melhor: Cap. 8 — Mais bom é melhor (9212 chars)
  - 10_nao_se_resolve_sozinho: Cap. 9 — Despovoamento não se resolve sozinho (9850 chars)
  - 11_controle_estatal: Cap. 10 — Controle estatal não pode forçar estabilização (8427 chars)
  - 12_dinheiro_e_resposta: Cap. 11 — Dinheiro é a resposta? (8851 chars)
  - 13_almeje_mais

## 6. Carregar OmniVoice (primeira vez baixa ~4GB de pesos)

In [6]:
from omnivoice import OmniVoice
import torch

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)
print("✓ Modelo carregado")


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, pleas

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

✓ Modelo carregado


## 7. Funções utilitárias

- `chunk_text` — quebra o texto em parágrafos (fallback em frases se passar de 700 chars).
- `synth_chunk` — gera wav 24kHz pra um trecho. Cache em 2 níveis (local + Drive) com fingerprint MD5 da voz no nome.
- `concat_wavs` — concatena tensores com `GAP_SEC` de silêncio entre eles.
- `wav_to_mp3` — exporta com ffmpeg (libmp3lame).


In [7]:
import re, hashlib, time, subprocess, shutil
import numpy as np
import torch
import torchaudio

SAMPLE_RATE_DEFAULT = 24000


def chunk_text(text: str, max_chars: int = 700) -> list[str]:
    """Quebra por \n\n; se um parágrafo passar de max_chars, quebra em sentenças."""
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    out = []
    for p in paras:
        if len(p) <= max_chars:
            out.append(p)
            continue
        sents = re.split(r'(?<=[.!?])\s+', p)
        buf = ""
        for s in sents:
            if len(buf) + len(s) + 1 > max_chars and buf:
                out.append(buf.strip())
                buf = s
            else:
                buf = (buf + " " + s).strip()
        if buf:
            out.append(buf.strip())
    return out


def text_hash(s: str, n: int = 12) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:n]


def _to_wav_tensor(out, default_sr=SAMPLE_RATE_DEFAULT):
    sr = default_sr
    if isinstance(out, (tuple, list)):
        if len(out) >= 2 and isinstance(out[1], (int, float)):
            audio, sr = out[0], int(out[1])
        else:
            audio = out[0]
    elif isinstance(out, dict):
        audio = out.get("audio", out.get("waveform", out.get("wav")))
        sr = int(out.get("sample_rate", out.get("sr", default_sr)))
    else:
        audio = out
    if isinstance(audio, np.ndarray):
        audio = torch.from_numpy(audio.astype("float32", copy=False))
    elif hasattr(audio, "cpu"):
        audio = audio.detach().cpu().float()
    if audio.ndim == 1:
        audio = audio.unsqueeze(0)
    return audio, sr


def synth_chunk(text: str, ref_audio: str = REF_AUDIO) -> tuple[torch.Tensor, int, str]:
    """Gera wav pra um chunk com cache em 2 níveis (local + Drive).
    Nome do cache inclui REF_AUDIO_TAG. Retorna (waveform, sr, hit_kind).
    """
    h = text_hash(text)
    name = f"chunk_{REF_AUDIO_TAG}_{h}.wav"
    local_path = os.path.join(LOCAL_WAV_CACHE, name)
    drive_path = os.path.join(DRIVE_WAV_CACHE, name)

    if os.path.exists(local_path):
        wf, sr = torchaudio.load(local_path)
        return wf, sr, "local"
    if os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
        wf, sr = torchaudio.load(local_path)
        return wf, sr, "drive"

    out = model.generate(text=text, ref_audio=ref_audio)
    wf, sr = _to_wav_tensor(out)
    torchaudio.save(local_path, wf, sr)
    return wf, sr, "miss"


def concat_wavs(wavs: list[tuple[torch.Tensor, int]], gap_sec: float = GAP_SEC) -> tuple[torch.Tensor, int]:
    sr = wavs[0][1]
    gap = torch.zeros((1, int(sr * gap_sec)), dtype=wavs[0][0].dtype)
    parts = []
    for i, (wf, s) in enumerate(wavs):
        assert s == sr, f"sample rate mismatch: {s} vs {sr}"
        parts.append(wf)
        if i != len(wavs) - 1:
            parts.append(gap)
    return torch.cat(parts, dim=1), sr


def wav_to_mp3(wav_path: str, mp3_path: str, bitrate: str = MP3_BITRATE):
    cmd = ["ffmpeg", "-y", "-i", wav_path, "-vn", "-codec:a", "libmp3lame", "-b:a", bitrate, mp3_path]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


print("✓ helpers prontos")


✓ helpers prontos


## 8. Gerar os 14 episódios

Cada episódio vira **um único MP3**. WAVs intermediários ficam cacheados em local **e** Drive (sobrevivem a reinício de sessão Colab).


In [8]:
import time

manifest = []
total_t0 = time.time()

for podcast in PODCASTS:
    pid = podcast["id"]
    title = podcast["title"]
    text = podcast["text"]
    chunks = chunk_text(text)
    print(f"\n=== {pid} — {title} ({len(chunks)} chunks) ===")

    podcast_t0 = time.time()
    wavs = []
    hits = {"local": 0, "drive": 0, "miss": 0}
    for i, c in enumerate(chunks, 1):
        wf, sr, hit = synth_chunk(c)
        dur = wf.shape[-1] / sr
        hits[hit] += 1
        flag = {"local": "[cache local]", "drive": "[cache drive]", "miss": "[gerou]"}[hit]
        preview = c[:70].replace(chr(10), " ")
        print(f"  [{i:02d}/{len(chunks):02d}] {dur:5.1f}s  {flag:14s}  {preview}{'...' if len(c)>70 else ''}")
        wavs.append((wf, sr))

    full_wav, sr = concat_wavs(wavs)
    total_dur = full_wav.shape[-1] / sr

    wav_path = os.path.join(LOCAL_OUT, f"{pid}.wav")
    mp3_path = os.path.join(LOCAL_OUT, f"{pid}.mp3")
    torchaudio.save(wav_path, full_wav, sr)
    wav_to_mp3(wav_path, mp3_path)

    shutil.copy(mp3_path, os.path.join(OUTPUT_DIR, f"{pid}.mp3"))

    elapsed = time.time() - podcast_t0
    size_mb = os.path.getsize(mp3_path) / (1024 * 1024)
    print(f"  ✓ {pid}.mp3 — {total_dur/60:.2f} min, {size_mb:.1f} MB, em {elapsed:.0f}s  "
          f"(hits: local={hits['local']} drive={hits['drive']} miss={hits['miss']})")
    manifest.append({
        "id": pid, "title": title,
        "duration_min": round(total_dur/60, 2),
        "size_mb": round(size_mb, 2),
        "chunks": len(chunks),
    })

# rsync local → Drive
t0 = time.time()
_ = os.popen(f"rsync -a --info=stats0 {LOCAL_WAV_CACHE}/ {DRIVE_WAV_CACHE}/").read()
print(f"\n✓ rsync cache local → Drive em {time.time()-t0:.1f}s")

total_min = sum(m["duration_min"] for m in manifest)
print(f"\n=== {len(manifest)} episódios — {total_min:.1f} min totais — wallclock {(time.time()-total_t0)/60:.1f} min ===")
import json
with open(os.path.join(LOCAL_OUT, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
shutil.copy(os.path.join(LOCAL_OUT, "manifest.json"), os.path.join(OUTPUT_DIR, "manifest.json"))



=== 01_prologo — Prólogo — o livro que pede pra você pensar de novo (18 chunks) ===


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

  [01/18]   3.5s  [gerou]         Episódio um. Prólogo: o livro que pede pra você pensar de novo.
  [02/18]   8.8s  [gerou]         Antes de tudo, uma frase que sustenta o livro inteiro: a humanidade es...
  [03/18]  36.1s  [gerou]         Pare por um segundo. Você, seus pais, seus avós, qualquer ancestral cu...
  [04/18]  16.6s  [gerou]         Esse é o ponto de partida desta série. Vamos percorrer o livro inteiro...
  [05/18]  39.8s  [gerou]         Quem são os autores. Dean Spears e Michael Geruso são economistas, pro...
  [06/18]  10.3s  [gerou]         E a tese desse livro nasceu de uma pergunta simples que eles passaram ...
  [07/18]  33.0s  [gerou]         A resposta deles é desconfortável pros dois lados do debate. Primeiro,...
  [08/18]   7.5s  [gerou]         Eles organizam o livro em torno de três afirmações grandes. Vale anota...
  [09/18]  30.7s  [gerou]         Afirmação um, da Parte um do livro: nenhum futuro é mais provável do q...
  [10/18]  27.0s  [gerou]         Afir

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [11/18]  23.9s  [gerou]         Afirmação três, da Parte quatro: ninguém ainda sabe como estabilizar u...
  [12/18]  36.0s  [gerou]         Tem um ponto sensível que merece ser dito agora, no episódio um, pra n...
  [13/18]  30.7s  [gerou]         Tem uma pergunta filosófica também, e essa é a mais difícil. Como você...
  [14/18]  36.9s  [gerou]         Pra fechar o prólogo, três coisas que eu acho que vale você levar pro ...
  [15/18]   4.9s  [gerou]         Spears e Geruso querem ocupar esse espaço com dados e com uma ética ma...
  [16/18]  21.6s  [gerou]         Segunda coisa: a tensão central do livro é essa. Ter filho é escolha p...
  [17/18]  23.3s  [gerou]         E terceira: o título do livro, After the Spike, depois do pico. A imag...
  [18/18]   7.9s  [gerou]         No próximo episódio, capítulo um, o pico em si. Como chegamos aqui, po...
  ✓ 01_prologo.mp3 — 6.76 min, 4.6 MB, em 226s  (hits: local=0 drive=0 miss=18)

=== 02_o_pico — Cap. 1 — O pico (20 chunks) ===
  [01/2

'/content/drive/MyDrive/AfterTheSpike_Podcasts/manifest.json'

## 9. Empacotar zip e baixar pro seu computador

In [9]:
import zipfile

zip_path = os.path.join(LOCAL_OUT, "after_the_spike_podcasts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(LOCAL_OUT)):
        if f.endswith(".mp3") or f == "manifest.json":
            zf.write(os.path.join(LOCAL_OUT, f), f)

shutil.copy(zip_path, os.path.join(OUTPUT_DIR, "after_the_spike_podcasts.zip"))
print(f"✓ zip: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)")
print(f"  cópia no Drive: {OUTPUT_DIR}/after_the_spike_podcasts.zip")

from google.colab import files
files.download(zip_path)


✓ zip: /content/podcasts_out/after_the_spike_podcasts.zip (79.9 MB)
  cópia no Drive: /content/drive/MyDrive/AfterTheSpike_Podcasts/after_the_spike_podcasts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. (Opcional) Preview rápido do primeiro episódio

In [10]:
from IPython.display import Audio, display
display(Audio(os.path.join(LOCAL_OUT, "01_prologo.mp3")))
